# Group 06 — Final MRI Quality Assessment

**Automated Image Quality Assessment for Cranial MRI Using AI Metrics**

Select **Runtime → Change runtime type → T4 GPU**, then run Steps 1–10 in order.
All source code is included. Advanced code is folded for readability; use **Show code**
to inspect it. No additional upgrade cells need to be pasted.

Workflow: **setup → MRI data → EDA → U-Net → baseline QA warm start → improved QA → test evaluation and analysis → image upload → PDF report → export**.
The two QA stages preserve the training method used in your supplied notebook.

Required results: **EDA missingness/correlation/distributions/count plots; before/after images, PSNR, SSIM, processing time, model accuracy, confusion matrices, ROC/AUC, loss curves, residual diagnostics, feature-importance plots, Good/Medium/Bad labels, and a PDF report**.
The upload step pauses Run all until you choose or cancel a file.

Old outputs were cleared because the supplied notebook mixed old-model test scores
with improved-model validation scores. Run this version to obtain consistent results.
This is a research prototype with synthetic, reference-SSIM-derived quality bands.

## 1. Install dependencies

Retain Colab’s PyTorch/CUDA; install only missing packages.

In [ ]:
# ONE-CELL MRI MODEL LOADER — no retraining required

from google.colab import drive
from pathlib import Path
import importlib.util
import subprocess
import sys
import json
import zipfile
import tempfile

drive.mount("/content/drive")

# Your existing extracted model and the folder containing your ZIP.
MODEL_FOLDER = "/content/drive/MyDrive/MRI_Model_jtaozq6z/MRI_Group06_Final"
ZIP_FOLDER = "/content/drive/MyDrive/MRI_IQA_FYP"

required = (
    "config.json",
    "enhancer_best.pt",
    "quality_best.pt",
    "source/mri_quality_final.py",
    "source/upload_mri.py",
)

def complete_model(folder):
    return all((folder / name).is_file() for name in required)

run_dir = Path(MODEL_FOLDER)
zip_folder = Path(ZIP_FOLDER)
saved_pointer = zip_folder / "quick_start_model_path.txt"

# Reuse a model extracted by this cell in an earlier session.
if not complete_model(run_dir) and saved_pointer.is_file():
    candidate = Path(saved_pointer.read_text(encoding="utf-8").strip())
    if complete_model(candidate):
        run_dir = candidate

# Extract only if the saved model cannot be found.
if not complete_model(run_dir):
    if not zip_folder.is_dir():
        raise FileNotFoundError(f"Drive folder not found: {zip_folder}")

    matches = []
    for archive_path in sorted(zip_folder.glob("*.zip")):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                names = set(archive.namelist())
                for name in sorted(names):
                    if name.endswith("enhancer_best.pt"):
                        prefix = name[:-len("enhancer_best.pt")]
                        if all(prefix + item in names for item in required):
                            matches.append((archive_path, prefix))
        except zipfile.BadZipFile:
            print("Skipping invalid ZIP:", archive_path.name)

    if not matches:
        raise FileNotFoundError(
            f"No complete exported model ZIP found in {zip_folder}. "
            "Upload the ZIP downloaded from Step 10."
        )

    if len(matches) == 1:
        archive_path, prefix = matches[0]
    else:
        print("\nAvailable models:")
        for index, (path, prefix_name) in enumerate(matches, 1):
            print(f"{index}. {path.name} — {prefix_name}")
        choice = int(input("Enter the model number: "))
        if not 1 <= choice <= len(matches):
            raise ValueError("Invalid model number.")
        archive_path, prefix = matches[choice - 1]

    destination = Path(
        tempfile.mkdtemp(prefix="MRI_Extracted_", dir=str(zip_folder))
    )

    print("Extracting:", archive_path.name)
    with zipfile.ZipFile(archive_path) as archive:
        destination_resolved = destination.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination_resolved):
                raise ValueError("ZIP contains an invalid file path.")
        archive.extractall(destination)

    run_dir = destination / prefix
    if not complete_model(run_dir):
        raise FileNotFoundError("Extracted model files are incomplete.")

    saved_pointer.write_text(str(run_dir), encoding="utf-8")

MODEL_FOLDER = str(run_dir)
print("Using model folder:", MODEL_FOLDER)

# Install only missing dependencies.
packages = {
    "numpy": "numpy>=1.26,<3",
    "cv2": "opencv-python-headless>=4.9,<5",
    "nibabel": "nibabel>=5.2,<6",
    "skimage": "scikit-image>=0.22,<1",
    "pandas": "pandas>=2.1,<4",
    "matplotlib": "matplotlib>=3.8,<4",
    "requests": "requests>=2.31,<3",
    "tqdm": "tqdm>=4.66,<5",
    "scipy": "scipy>=1.11,<2",
    "sklearn": "scikit-learn>=1.4,<2",
    "torch": "torch>=2.2,<3",
    "reportlab": "reportlab>=4,<5",
    "PIL": "Pillow",
}

missing_packages = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("Installing missing dependencies...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

# Load the project definitions from your own exported model ZIP.
FINAL_MODEL_SOURCE = (
    run_dir / "source/mri_quality_final.py"
).read_text(encoding="utf-8")

exec(
    compile(FINAL_MODEL_SOURCE, "mri_quality_final.py", "exec"),
    globals(),
)

# Restore the trained enhancement and quality-assessment networks.
cfg = Config(**json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
))
pipeline = MRIPipeline.from_folder(run_dir)
device = pipeline.device
enhancer = pipeline.enhancer
quality_model = pipeline.quality_model

FINAL_STEP_SOURCES = {
    path.name: path.read_text(encoding="utf-8")
    for path in (run_dir / "source").glob("*.py")
    if path.name != "mri_quality_final.py"
}

# Restore saved evaluation results, when available.
QA_TEST_METRICS = {}
provenance = run_dir / "qa_test_provenance.json"
if provenance.is_file():
    QA_TEST_METRICS = json.loads(
        provenance.read_text(encoding="utf-8")
    ).get("metrics", {})

last_upload_result = None

print(f"\nModel loaded successfully on {device}.")
print(f"Input resolution: {pipeline.image_size} × {pipeline.image_size}")
print("No training or dataset download is needed.")
print("\nChoose an MRI image below.")

# Open the uploader and analyze your MRI.
exec(
    compile(
        FINAL_STEP_SOURCES["upload_mri.py"],
        "upload_mri.py",
        "exec",
    ),
    globals(),
)

# Generate and download the PDF if the report source was exported.
if last_upload_result is not None:
    report_source = FINAL_STEP_SOURCES.get("uploaded_mri_report.py")
    if report_source:
        exec(
            compile(report_source, "uploaded_mri_report.py", "exec"),
            globals(),
        )
    else:
        print("MRI analysis completed. PDF source was not included in the ZIP.")

In [ ]:
import importlib.util
import subprocess
import sys

requirements = {
    "numpy": "numpy>=1.26,<3", "cv2": "opencv-python-headless>=4.9,<5",
    "nibabel": "nibabel>=5.2,<6", "skimage": "scikit-image>=0.22,<1",
    "pandas": "pandas>=2.1,<4", "matplotlib": "matplotlib>=3.8,<4",
    "requests": "requests>=2.31,<3", "tqdm": "tqdm>=4.66,<5", "scipy": "scipy>=1.11,<2", "sklearn": "scikit-learn>=1.4,<2",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch>=2.2,<3"])
print("Dependencies ready.")

## 2. Load project functions and choose settings

Run both cells. All shared definitions load before training and evaluation.
Use **standard** for the 256 × 256 project model and **quick** for a 128 × 128 workflow check. Standard training uses batch size 4 to limit GPU memory. Enable Drive
before training if checkpoints must survive a runtime reset.

In [ ]:
#@title 2a. Load project functions (advanced code)
from pathlib import Path
FINAL_MODEL_SOURCE = r'''
"""Group 06: cranial MRI enhancement and quality assessment.

The Colab notebook embeds this module, so uploading the notebook alone is enough.
"""
# %% imports
from __future__ import annotations

import dataclasses
import hashlib
import json
import math
import random
import re
import time
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from urllib.parse import quote
import xml.etree.ElementTree as ET

import cv2
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import requests
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm.auto import tqdm


@dataclass
class Config:
    # "quick" verifies the workflow; "standard" is the project experiment.
    profile: str = "standard"
    dataset_id: str = "ds000221"
    n_subjects: int = 20
    image_size: int = 256  # Preserve more spatial detail than the old 128x128 setting.
    slices_per_subject: int = 48
    epochs: int = 25
    qa_epochs: int = 12
    batch_size: int = 4  # Smaller batches keep 256x256 U-Net training practical on Colab GPUs.
    base_channels: int = 24
    learning_rate: float = 1e-3
    patience: int = 7
    seed: int = 42
    num_workers: int = 0
    noise_kind: str = "mixed"  # gaussian, rician, or mixed
    root: str = "/content/mri_group06"
    local_nifti_dir: str = ""  # optional BIDS NIfTI data instead of download
    resume: bool = True

    @classmethod
    def for_profile(cls, profile="standard", root="/content/mri_group06"):
        cfg = cls(profile=profile, root=str(root))
        if profile == "quick":
            cfg.n_subjects, cfg.slices_per_subject = 10, 16
            cfg.image_size, cfg.batch_size = 128, 8
            cfg.epochs, cfg.qa_epochs, cfg.base_channels = 3, 3, 16
        elif profile != "standard":
            raise ValueError("Choose 'quick' or 'standard'.")
        return cfg


def setup(cfg):
    assert cfg.image_size >= 32 and cfg.image_size % 8 == 0
    assert cfg.n_subjects >= 10 and cfg.n_subjects % 10 == 0, "Use 10, 20, 30, ... subjects for an exact 70/30 split."
    assert cfg.noise_kind in ("gaussian", "rician", "mixed")
    assert cfg.epochs > 0 and cfg.qa_epochs > 0
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    root = Path(cfg.root)
    root.mkdir(parents=True, exist_ok=True)
    print("Device:", device, "| PyTorch:", torch.__version__)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("CPU detected. Select a GPU in Colab for the standard experiment.")
    return device


# %% download
S3 = "https://s3.amazonaws.com/openneuro.org"
S3NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}


def http_session():
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    session = requests.Session()
    retry = Retry(total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


def list_s3_entries(session, prefix, delimiter=None):
    objects, prefixes, token = [], [], None
    for _ in range(1000):
        params = {"list-type": "2", "prefix": prefix, "max-keys": 1000}
        if delimiter:
            params["delimiter"] = delimiter
        if token:
            params["continuation-token"] = token
        response = session.get(S3, params=params, timeout=(15, 120))
        response.raise_for_status()
        tree = ET.fromstring(response.content)
        prefixes.extend(item.findtext("s3:Prefix", namespaces=S3NS)
                        for item in tree.findall("s3:CommonPrefixes", S3NS))
        for obj in tree.findall("s3:Contents", S3NS):
            key = obj.findtext("s3:Key", namespaces=S3NS)
            if key:
                objects.append({"key": key, "size": int(obj.findtext("s3:Size", namespaces=S3NS)),
                                "etag": obj.findtext("s3:ETag", namespaces=S3NS)})
        if tree.findtext("s3:IsTruncated", namespaces=S3NS) != "true":
            break
        token = tree.findtext("s3:NextContinuationToken", namespaces=S3NS)
        if not token:
            raise RuntimeError("Incomplete S3 listing: missing continuation token.")
    else:
        raise RuntimeError("Dataset listing exceeded safety limit.")
    return objects, sorted(prefixes)


def list_t1_objects(dataset_id, subject_limit=20, seed=42):
    """Sample subject folders first, avoiding a costly listing of all derivatives."""
    if not re.fullmatch(r"ds\d{6}", dataset_id):
        raise ValueError("Expected an OpenNeuro ID such as ds000221.")
    session = http_session()
    _, subject_prefixes = list_s3_entries(session, dataset_id + "/sub-", delimiter="/")
    objects, found = [], 0
    for prefix in np.random.default_rng(seed).permutation(subject_prefixes):
        entries, _ = list_s3_entries(session, str(prefix))
        matches = [row for row in entries if row["key"].endswith(("_T1w.nii.gz", "_T1w.nii"))
                   and "/anat/" in row["key"] and "/derivatives/" not in row["key"]]
        if matches:
            for row in matches:
                row["subject"] = re.search(r"/(sub-[^/]+)/", row["key"]).group(1)
            objects.extend(matches)
            found += 1
        if found >= subject_limit:
            break
    if not objects:
        raise RuntimeError("No T1w NIfTI scans found. Check dataset access or set local_nifti_dir.")
    return sorted(objects, key=lambda row: row["key"])


def download_file(session, key, destination, expected_size=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and (expected_size is None or destination.stat().st_size == expected_size):
        return destination
    temporary = destination.with_name(destination.name + ".part")
    url = S3 + "/" + quote(key, safe="/")
    with session.get(url, stream=True, timeout=(15, 180)) as response:
        response.raise_for_status()
        with temporary.open("wb") as handle:
            for block in response.iter_content(1024 * 1024):
                if block:
                    handle.write(block)
    if expected_size is not None and temporary.stat().st_size != expected_size:
        raise IOError(f"Incomplete download: {key}; rerun this cell to retry.")
    temporary.replace(destination)
    return destination


def acquire_dataset(cfg):
    data_dir = Path(cfg.root) / "data" / cfg.dataset_id
    data_dir.mkdir(parents=True, exist_ok=True)
    if cfg.local_nifti_dir:
        candidates = []
        for file in sorted(Path(cfg.local_nifti_dir).rglob("*")):
            if file.name.endswith(("_T1w.nii.gz", "_T1w.nii")) and "derivatives" not in file.parts:
                subject = next((p for p in file.parts if p.startswith("sub-")), None)
                if subject:
                    candidates.append({"subject": subject, "key": str(file.resolve()),
                                       "size": file.stat().st_size, "etag": "local", "path": str(file)})
    else:
        listing_path = data_dir / f"t1_listing_seed{cfg.seed}_n{cfg.n_subjects}.json"
        if listing_path.exists():
            candidates = json.loads(listing_path.read_text())
        else:
            candidates = list_t1_objects(cfg.dataset_id, cfg.n_subjects, cfg.seed)
            listing_path.write_text(json.dumps(candidates, indent=2))
    # Select exactly one acquisition per person; never split repeated scans across sets.
    per_subject = {}
    for row in sorted(candidates, key=lambda row: row["key"]):
        per_subject.setdefault(row["subject"], row)
    subjects = np.array(sorted(per_subject))
    if len(subjects) < cfg.n_subjects:
        raise ValueError(f"Need {cfg.n_subjects} distinct subjects; found {len(subjects)}.")
    subjects = np.random.default_rng(cfg.seed).permutation(subjects)[:cfg.n_subjects]
    records, session = [], http_session()
    for subject in tqdm(subjects, desc="Download T1 MRI subjects"):
        record = dict(per_subject[subject])
        if not cfg.local_nifti_dir:
            filename = Path(record["key"]).name
            file = download_file(session, record["key"], data_dir / subject / filename, record["size"])
            record["path"] = str(file)
        # Force a header read, catching HTML/error payloads and broken downloads now.
        image = nib.load(record["path"])
        if len(image.shape) != 3:
            raise ValueError(f"Expected a 3D anatomical MRI: {record['path']} {image.shape}")
        records.append(record)
    if not cfg.local_nifti_dir:
        for name in ("dataset_description.json", "README", "CHANGES", "participants.tsv"):
            try:
                download_file(session, cfg.dataset_id + "/" + name, data_dir / name)
            except requests.HTTPError as error:
                if error.response.status_code != 404:
                    raise
    print(f"Using {len(records)} subjects from {cfg.dataset_id}; source files stay outside result ZIP.")
    return records


# %% preprocessing
def normalize_volume(volume):
    volume = np.nan_to_num(np.asarray(volume, dtype=np.float32), nan=0, posinf=0, neginf=0)
    positive = volume[volume > 0]
    if positive.size < 100:
        raise ValueError("Volume has too few positive voxels for T1 intensity normalization.")
    low, high = np.percentile(positive, [1, 99])
    if high <= low:
        raise ValueError("MRI intensity range is constant.")
    return np.clip((volume - low) / (high - low), 0, 1).astype(np.float32)


def resize_with_padding(image, image_size):
    """Resize a grayscale slice without warping its aspect ratio; pad with MRI background."""
    image = np.asarray(image, dtype=np.float32)
    if image.ndim != 2 or not image.size:
        raise ValueError("Expected a nonempty 2D grayscale MRI slice.")
    height, width = image.shape
    scale = min(image_size / height, image_size / width)
    new_h = max(1, min(image_size, int(round(height * scale))))
    new_w = max(1, min(image_size, int(round(width * scale))))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(image, (new_w, new_h), interpolation=interpolation)
    canvas = np.zeros((image_size, image_size), dtype=np.float32)
    top, left = (image_size - new_h) // 2, (image_size - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized
    return np.clip(canvas, 0, 1)


def split_subjects(records, cfg):
    # Input order was randomized at acquisition. Keep this split in the manifest.
    n_development = len(records) * 7 // 10
    development, test = records[:n_development], records[n_development:]
    n_validation = max(1, int(round(len(development) * 0.2)))
    splits = {"train": development[:-n_validation], "validation": development[-n_validation:], "test": test}
    ids = {name: {row["subject"] for row in rows} for name, rows in splits.items()}
    assert not (ids["train"] & ids["validation"] or ids["train"] & ids["test"] or ids["validation"] & ids["test"])
    assert len(development) / len(records) == 0.7
    print("70% development (training + internal validation), 30% held-out test:")
    print({name: len(rows) for name, rows in splits.items()})
    return splits


def prepare_data(records, cfg):
    splits = split_subjects(records, cfg)
    # Isolate incompatible experiments instead of silently reusing their checkpoints.
    signature = {key: value for key, value in asdict(cfg).items()
                 if key not in ("root", "resume", "epochs", "qa_epochs", "num_workers", "local_nifti_dir")}
    signature["subjects"] = {name: [{k: row[k] for k in ("subject", "key", "size", "etag")} for row in rows]
                             for name, rows in splits.items()}
    run_id = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:12]
    run_dir = Path(cfg.root) / "runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
    (run_dir / "subject_manifest.json").write_text(json.dumps(splits, indent=2))
    (run_dir / "experiment_signature.json").write_text(json.dumps(signature, indent=2))
    data = {}
    for split, rows in splits.items():
        images, metadata = [], []
        for row in tqdm(rows, desc=f"Prepare {split}"):
            image = nib.as_closest_canonical(nib.load(row["path"]))
            volume = normalize_volume(image.get_fdata(dtype=np.float32))
            # Canonical NIfTI: axis 2 is the superior/inferior (axial) axis.
            depth = volume.shape[2]
            valid = [z for z in range(int(depth * 0.15), int(depth * 0.85))
                     if np.mean(volume[:, :, z] > 0.08) >= 0.12]
            if len(valid) < cfg.slices_per_subject:
                raise ValueError(f"Too few nonempty axial slices in {row['subject']}: {len(valid)}. Reduce slices_per_subject.")
            chosen = np.array(valid)[np.linspace(0, len(valid) - 1, cfg.slices_per_subject).round().astype(int)]
            assert len(set(chosen)) == cfg.slices_per_subject
            for z in chosen:
                # Transpose/flip is only for consistent radiological display, applied to every pair.
                axial = np.flipud(volume[:, :, z].T)
                resized = resize_with_padding(axial, cfg.image_size)
                images.append(np.ascontiguousarray(resized, dtype=np.float32))
                metadata.append({"subject": row["subject"], "slice": int(z), "source": row["key"]})
        data[split] = {"images": np.stack(images), "metadata": metadata}
    print("Slices:", {name: len(values["images"]) for name, values in data.items()})
    print("Result folder:", run_dir)
    return data, run_dir


SCENARIOS = {"mild": (2, 0.03), "moderate": (3, 0.06), "severe": (4, 0.10)}


def degrade(clean, rng, noise_kind="rician", scale=None, sigma=None):
    scale = int(rng.choice([2, 3, 4])) if scale is None else int(scale)
    sigma = float(rng.uniform(0.02, 0.12)) if sigma is None else float(sigma)
    kind = str(rng.choice(["gaussian", "rician"])) if noise_kind == "mixed" else noise_kind
    h, w = clean.shape
    small = cv2.resize(clean, (max(8, w // scale), max(8, h // scale)), interpolation=cv2.INTER_AREA)
    blurred = np.clip(cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC), 0, 1)
    n1 = rng.normal(0, sigma, clean.shape)
    if kind == "rician":
        n2 = rng.normal(0, sigma, clean.shape)
        degraded = np.sqrt((blurred + n1) ** 2 + n2 ** 2)
    elif kind == "gaussian":
        degraded = blurred + n1
    else:
        raise ValueError("Noise must be gaussian, rician, or mixed.")
    return np.clip(degraded, 0, 1).astype(np.float32)


class MRIPairs(Dataset):
    def __init__(self, images, cfg, training=False, seed_offset=0):
        self.images, self.cfg, self.training = images, cfg, training
        self.seed_offset, self.epoch = seed_offset, 0

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        seed = self.cfg.seed + self.seed_offset + index + (self.epoch * 100003 if self.training else 0)
        rng = np.random.default_rng(seed)
        clean = self.images[index].copy()
        if self.training:
            clean = np.rot90(clean, int(rng.integers(4)))
            if rng.random() < 0.5:
                clean = np.fliplr(clean)
        bad = degrade(clean, rng, self.cfg.noise_kind)
        return torch.from_numpy(bad[None]), torch.from_numpy(np.ascontiguousarray(clean[None]))


def make_pair_loaders(data, cfg):
    datasets = {name: MRIPairs(data[name]["images"], cfg, name == "train", 0 if name == "train" else 200000)
                for name in ("train", "validation")}
    # Bump the signature when preprocessing or the enhancement objective changes, so
    # an older checkpoint cannot silently resume under the new training setup.
    for ds in datasets.values():
        ds.input_signature = f"paired-mri-v2-{cfg.image_size}-{cfg.noise_kind}-edge-loss"
    return {name: DataLoader(ds, batch_size=cfg.batch_size, shuffle=name == "train",
                             num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
            for name, ds in datasets.items()}


# %% models
class ConvBlock(nn.Module):
    def __init__(self, incoming, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming, outgoing, 3, padding=1),
                                    nn.GroupNorm(4, outgoing), nn.ReLU(inplace=True),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1),
                                    nn.GroupNorm(4, outgoing), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.layers(x)


class UNet(nn.Module):
    def __init__(self, base=24):
        super().__init__()
        self.e1, self.e2, self.e3 = ConvBlock(1, base), ConvBlock(base, base*2), ConvBlock(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.bridge = ConvBlock(base*4, base*8)
        self.up3, self.up2, self.up1 = (nn.ConvTranspose2d(base*8, base*4, 2, stride=2),
                                       nn.ConvTranspose2d(base*4, base*2, 2, stride=2),
                                       nn.ConvTranspose2d(base*2, base, 2, stride=2))
        self.d3, self.d2, self.d1 = ConvBlock(base*8, base*4), ConvBlock(base*4, base*2), ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
        # Start with the input image and learn a bounded correction.
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.pool(a))
        c = self.e3(self.pool(b))
        middle = self.bridge(self.pool(c))
        z = self.d3(torch.cat([self.up3(middle), c], dim=1))
        z = self.d2(torch.cat([self.up2(z), b], dim=1))
        z = self.d1(torch.cat([self.up1(z), a], dim=1))
        return (x + 0.5 * torch.tanh(self.out(z))).clamp(0, 1)


def differentiable_ssim(x, y):
    # Gaussian 11x11 SSIM, data_range=1. Evaluation uses scikit-image's Wang settings.
    coord = torch.arange(11, device=x.device, dtype=x.dtype) - 5
    g = torch.exp(-(coord ** 2) / (2 * 1.5 ** 2))
    g = g / g.sum()
    kernel = (g[:, None] * g[None, :])[None, None]
    blur = lambda value: torch.nn.functional.conv2d(value, kernel)
    mx, my = blur(x), blur(y)
    vx = (blur(x*x) - mx*mx).clamp_min(0)
    vy = (blur(y*y) - my*my).clamp_min(0)
    covariance = blur(x*y) - mx*my
    value = ((2*mx*my + 0.01**2) * (2*covariance + 0.03**2)) / (
        (mx*mx + my*my + 0.01**2) * (vx + vy + 0.03**2))
    return value.mean()


def enhancement_loss(prediction, target):
    """Balance pixel fidelity, structure, and edge retention to limit over-smoothing."""
    pixel = nn.functional.l1_loss(prediction, target)
    structure = 1 - differentiable_ssim(prediction, target)
    pred_dx, target_dx = prediction[..., 1:] - prediction[..., :-1], target[..., 1:] - target[..., :-1]
    pred_dy, target_dy = prediction[..., 1:, :] - prediction[..., :-1, :], target[..., 1:, :] - target[..., :-1, :]
    edges = 0.5 * (nn.functional.l1_loss(pred_dx, target_dx) + nn.functional.l1_loss(pred_dy, target_dy))
    return 0.60 * pixel + 0.30 * structure + 0.10 * edges


class QualityNet(nn.Module):
    """No-reference, synthetic-domain PSNR/SSIM regressor; not a clinical quality classifier."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4)))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64*4*4, 64), nn.ReLU(), nn.Linear(64, 2), nn.Sigmoid())

    def forward(self, x):
        return self.head(self.features(x))


def fit_model(model, loaders, device, cfg, run_dir, name, epochs, loss_fn):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    best, start, stale, history = float("inf"), 0, 0, []
    last_path, best_path = Path(run_dir) / f"{name}_last.pt", Path(run_dir) / f"{name}_best.pt"
    input_signature = getattr(loaders["train"].dataset, "input_signature", "paired-mri")
    if cfg.resume and last_path.exists():
        # Only load this project's own checkpoint files. Never load untrusted pickle checkpoints.
        checkpoint = torch.load(last_path, map_location=device, weights_only=False)
        if checkpoint.get("input_signature") == input_signature:
            model.load_state_dict(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            start, best, stale = checkpoint["epoch"] + 1, checkpoint["best"], checkpoint["stale"]
            history = checkpoint["history"]
            torch.set_rng_state(checkpoint["torch_rng"].cpu())
            if device.type == "cuda" and checkpoint["cuda_rng"] is not None:
                torch.cuda.set_rng_state_all([state.cpu() for state in checkpoint["cuda_rng"]])
            print(f"Resuming {name} at epoch {start + 1}")
        else:
            print(f"{name}: training inputs changed; starting a fresh model instead of resuming stale weights.")
    for epoch in range(start, epochs):
        if stale >= cfg.patience:
            print(f"{name}: early stopping already reached; loading best validation checkpoint.")
            break
        started = time.perf_counter()
        losses = {}
        for phase in ("train", "validation"):
            training = phase == "train"
            model.train(training)
            if hasattr(loaders[phase].dataset, "epoch"):
                loaders[phase].dataset.epoch = epoch
            total, count = 0.0, 0
            for inputs, targets in loaders[phase]:
                inputs, targets = inputs.to(device), targets.to(device)
                optimizer.zero_grad(set_to_none=True)
                with torch.set_grad_enabled(training):
                    predictions = model(inputs)
                    loss = loss_fn(predictions, targets)
                    if not torch.isfinite(loss):
                        raise RuntimeError(f"Non-finite {name} loss.")
                    if training:
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                total += loss.item() * len(inputs)
                count += len(inputs)
            losses[phase] = total / count
        if device.type == "cuda":
            torch.cuda.synchronize()
        improved = losses["validation"] < best - 1e-6
        if improved:
            best, stale = losses["validation"], 0
            torch.save({"model": model.state_dict(), "config": asdict(cfg), "epoch": epoch}, best_path)
        else:
            stale += 1
        row = {"epoch": epoch + 1, "train_loss": losses["train"], "validation_loss": losses["validation"],
               "seconds": time.perf_counter() - started}
        history.append(row)
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": epoch,
                    "best": best, "stale": stale, "history": history, "torch_rng": torch.get_rng_state(),
                    "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
                    "input_signature": input_signature}, last_path)
        pd.DataFrame(history).to_csv(Path(run_dir) / f"{name}_history.csv", index=False)
        print(f"{name} {epoch+1:02d}/{epochs}: train={losses['train']:.5f}, val={losses['validation']:.5f}, {row['seconds']:.1f}s")
    if not best_path.exists():
        raise RuntimeError("No trained checkpoint available. Set epochs to at least 1.")
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)["model"])
    model.eval()
    return model, pd.DataFrame(history)


def plot_history(history, title, path):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history.epoch, history.train_loss, label="Training")
    ax.plot(history.epoch, history.validation_loss, label="Validation")
    ax.set(xlabel="Epoch", ylabel="Loss", title=title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()
    plt.close(fig)


# %% quality
def true_metrics(reference, candidate):
    if reference.shape != candidate.shape or reference.ndim != 2:
        raise ValueError("Metrics require aligned 2D arrays of identical shape.")
    reference, candidate = np.asarray(reference, np.float32), np.asarray(candidate, np.float32)
    psnr = peak_signal_noise_ratio(reference, candidate, data_range=1.0)
    ssim = structural_similarity(reference, candidate, data_range=1.0, gaussian_weights=True,
                                 sigma=1.5, use_sample_covariance=False)
    # Foreground is intensity-derived head tissue; this is not an anatomical brain segmentation.
    mask = reference > 0.08
    mse = float(np.mean((reference[mask] - candidate[mask]) ** 2)) if mask.any() else float("nan")
    foreground = float(-10 * np.log10(mse)) if mse > 0 else (float("inf") if mse == 0 else float("nan"))
    return {"psnr_db": float(psnr), "ssim": float(ssim), "foreground_psnr_db": foreground}


@torch.inference_mode()
def enhance_array(image, model, device):
    tensor = torch.from_numpy(np.ascontiguousarray(image[None, None], dtype=np.float32)).to(device)
    return model(tensor)[0, 0].cpu().numpy()


def make_qa_dataset(split_data, enhancer, device, cfg, seed_offset):
    inputs, targets = [], []
    for index, clean in enumerate(tqdm(split_data["images"], desc="Prepare quality-estimator labels")):
        rng = np.random.default_rng(cfg.seed + seed_offset + index)
        degraded = degrade(clean, rng, cfg.noise_kind)
        enhanced = enhance_array(degraded, enhancer, device)
        for candidate in (degraded, enhanced):
            metrics = true_metrics(clean, candidate)
            inputs.append(candidate[None])
            targets.append([np.clip(metrics["psnr_db"], 0, 60) / 60,
                            (np.clip(metrics["ssim"], -1, 1) + 1) / 2])
    return torch.utils.data.TensorDataset(torch.from_numpy(np.stack(inputs).astype(np.float32)),
                                          torch.tensor(targets, dtype=torch.float32))


def make_qa_loaders(data, enhancer, device, cfg):
    signature = hashlib.sha256()
    for name, value in enhancer.state_dict().items():
        signature.update(name.encode())
        signature.update(value.detach().cpu().numpy().tobytes())
    loaders = {}
    for name, offset in (("train", 400000), ("validation", 500000)):
        dataset = make_qa_dataset(data[name], enhancer, device, cfg, offset)
        dataset.input_signature = signature.hexdigest()
        loaders[name] = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=name == "train", num_workers=0)
    return loaders


@torch.inference_mode()
def estimate_quality(image, qa_model, device):
    x = torch.from_numpy(np.ascontiguousarray(image[None, None], dtype=np.float32)).to(device)
    result = qa_model(x)[0].cpu().numpy()
    return {"estimated_psnr_db": float(result[0] * 60), "estimated_ssim": float(result[1] * 2 - 1)}


# %% evaluation
def timed_call(callable_fn, device=None):
    if device is not None and device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    result = callable_fn()
    if device is not None and device.type == "cuda":
        torch.cuda.synchronize()
    return result, (time.perf_counter() - start) * 1000


def evaluate(data, enhancer, qa_model, device, cfg, run_dir):
    # Fixed unseen subjects and fixed degradation seeds. No tuning on these results.
    rows, qa_rows, examples = [], [], {}
    kinds = ("gaussian", "rician") if cfg.noise_kind == "mixed" else (cfg.noise_kind,)
    dummy = data["test"]["images"][0]
    for _ in range(3):
        enhance_array(dummy, enhancer, device)
        estimate_quality(dummy, qa_model, device)
    chosen_indices = [len(data["test"]["images"]) // 2]
    for kind_index, kind in enumerate(kinds):
        for severity_index, (severity, (scale, sigma)) in enumerate(SCENARIOS.items()):
            for index, clean in enumerate(tqdm(data["test"]["images"], desc=f"Test {kind}/{severity}")):
                rng = np.random.default_rng(cfg.seed + 700000 + kind_index*100000 + severity_index*10000 + index)
                bad = degrade(clean, rng, kind, scale, sigma)
                smooth, smooth_ms = timed_call(lambda: cv2.GaussianBlur(bad, (3, 3), 0.7))
                restored, model_ms = timed_call(lambda: enhance_array(bad, enhancer, device), device)
                outputs = {"Before (degraded)": (bad, 0.0), "Gaussian baseline": (smooth, smooth_ms),
                           "After (U-Net)": (restored, model_ms)}
                common = {**data["test"]["metadata"][index], "noise": kind, "severity": severity,
                          "scale": scale, "sigma": sigma}
                for method, (candidate, milliseconds) in outputs.items():
                    metrics = true_metrics(clean, candidate)
                    rows.append({**common, "method": method, **metrics, "processing_ms": milliseconds})
                    if method != "Gaussian baseline":
                        estimates, qa_ms = timed_call(lambda: estimate_quality(candidate, qa_model, device), device)
                        qa_rows.append({**common, "method": method, **metrics, **estimates,
                                        "qa_processing_ms": qa_ms})
                if index in chosen_indices and kind == kinds[-1]:
                    examples[severity] = {"reference": clean, "before": bad, "baseline": smooth,
                                          "after": restored, "noise": kind, "metadata": common}
    results, quality = pd.DataFrame(rows), pd.DataFrame(qa_rows)
    results.to_csv(Path(run_dir) / "per_slice_metrics.csv", index=False)
    quality.to_csv(Path(run_dir) / "quality_predictions.csv", index=False)
    summary = results.groupby(["noise", "severity", "method"], sort=False).agg(
        n=("ssim", "size"), psnr_mean_db=("psnr_db", "mean"), psnr_std_db=("psnr_db", "std"),
        ssim_mean=("ssim", "mean"), ssim_std=("ssim", "std"),
        foreground_psnr_mean_db=("foreground_psnr_db", "mean"),
        mean_ms=("processing_ms", "mean"), median_ms=("processing_ms", "median"),
        p95_ms=("processing_ms", lambda x: float(np.percentile(x, 95)))).reset_index()
    summary.to_csv(Path(run_dir) / "evaluation_summary.csv", index=False)
    subjects = results.groupby(["subject", "noise", "severity", "method"], sort=False)[["psnr_db", "ssim"]].mean().reset_index()
    subjects.to_csv(Path(run_dir) / "subject_metrics.csv", index=False)
    paired = subjects.pivot(index=["subject", "noise", "severity"], columns="method", values=["psnr_db", "ssim"])
    gains = pd.DataFrame({"psnr_gain_db": paired[("psnr_db", "After (U-Net)")] - paired[("psnr_db", "Before (degraded)")],
                          "ssim_gain": paired[("ssim", "After (U-Net)")] - paired[("ssim", "Before (degraded)")]})
    gains.reset_index().to_csv(Path(run_dir) / "before_after_subject_gains.csv", index=False)
    qa_report = []
    for method, group in quality.groupby("method"):
        for metric, estimated in (("psnr_db", "estimated_psnr_db"), ("ssim", "estimated_ssim")):
            error = group[estimated] - group[metric]
            qa_report.append({"method": method, "metric": metric, "mae": float(error.abs().mean()),
                              "rmse": float(np.sqrt(np.mean(error**2))),
                              "spearman": float(group[estimated].corr(group[metric], method="spearman"))})
    qa_report = pd.DataFrame(qa_report)
    qa_report.to_csv(Path(run_dir) / "quality_estimator_accuracy.csv", index=False)
    mean_gains = gains.mean().to_dict()
    if mean_gains["psnr_gain_db"] <= 0 or mean_gains["ssim_gain"] <= 0:
        print("The model did not improve both average metrics in this run. Report the measured results; quick runs are not final experiments.")
    print("Mean subject-level before/after gains:", mean_gains)
    return results, summary, quality, qa_report, examples


def plot_comparisons(examples, run_dir):
    fig, axes = plt.subplots(len(examples), 5, figsize=(17, 3.5*len(examples)), squeeze=False)
    for row, (severity, example) in enumerate(examples.items()):
        panels = [("Reference MRI", example["reference"]), ("Before: simulated low quality", example["before"]),
                  ("Gaussian baseline", example["baseline"]), ("After: U-Net", example["after"]),
                  ("Absolute U-Net error", np.abs(example["after"] - example["reference"]))]
        for column, (label, image) in enumerate(panels):
            ax = axes[row, column]
            ax.imshow(image, cmap="magma" if column == 4 else "gray", vmin=0, vmax=0.25 if column == 4 else 1)
            title = label
            if column == 0:
                title += f"\n{example['noise']} / {severity}"
            if 0 < column < 4:
                values = true_metrics(example["reference"], image)
                title += f"\nPSNR {values['psnr_db']:.2f} dB | SSIM {values['ssim']:.3f}"
            ax.set_title(title, fontsize=10)
            ax.axis("off")
        for name in ("reference", "before", "baseline", "after"):
            cv2.imwrite(str(Path(run_dir) / f"example_{severity}_{name}.png"), (example[name]*255).round().astype(np.uint8))
    fig.suptitle("Fixed held-out example: same subject and slice across severity levels", fontsize=14)
    fig.tight_layout()
    fig.savefig(Path(run_dir) / "before_after_comparison.png", dpi=160)
    plt.show()
    plt.close(fig)


def plot_metric_summary(results, run_dir):
    import inspect
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    order = ["Before (degraded)", "Gaussian baseline", "After (U-Net)"]
    for ax, metric, title in zip(axes, ["psnr_db", "ssim", "processing_ms"], ["PSNR (dB)", "SSIM", "Processing time (ms/image)"]):
        groups = [results.loc[results.method == method, metric].values for method in order]
        labels = ["Before", "Gaussian", "U-Net"]
        keyword = "tick_labels" if "tick_labels" in inspect.signature(ax.boxplot).parameters else "labels"
        ax.boxplot(groups, **{keyword: labels}, showfliers=False)
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(Path(run_dir) / "metrics_and_processing_time.png", dpi=150)
    plt.show()
    plt.close(fig)


def plot_quality_accuracy(quality, run_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, actual, estimated, label in zip(axes, ["psnr_db", "ssim"], ["estimated_psnr_db", "estimated_ssim"], ["PSNR (dB)", "SSIM"]):
        ax.scatter(quality[actual], quality[estimated], s=6, alpha=0.25)
        low = min(quality[actual].min(), quality[estimated].min())
        high = max(quality[actual].max(), quality[estimated].max())
        ax.plot([low, high], [low, high], "r--", label="Perfect prediction")
        ax.set(xlabel=f"Measured {label}", ylabel=f"Estimated {label}")
        ax.legend()
    fig.tight_layout()
    fig.savefig(Path(run_dir) / "quality_estimator_accuracy.png", dpi=150)
    plt.show()
    plt.close(fig)


# %% inference
def quality_band(estimated_ssim, medium=0.60, good=0.80):
    """Project bands on estimated SSIM; these are not radiologist quality labels."""
    if not 0 <= medium < good <= 1:
        raise ValueError("Thresholds must satisfy 0 <= medium < good <= 1.")
    score = float(estimated_ssim)
    if not math.isfinite(score) or not -1 <= score <= 1:
        raise ValueError("Estimated SSIM must be finite and between -1 and 1.")
    return "Good" if score >= good else "Medium" if score >= medium else "Bad"


def label_quality_report(report, medium=0.60, good=0.80):
    """Add before/after quality labels without mutating the original report."""
    labeled = dict(report)
    for key in ("before_estimated", "after_estimated"):
        estimates = dict(report[key])
        estimates["quality_label"] = quality_band(estimates["estimated_ssim"], medium, good)
        labeled[key] = estimates
    labeled["quality_bands"] = {"basis": "Estimated SSIM", "medium_at_or_above": medium,
                                "good_at_or_above": good,
                                "scope": "Editable project thresholds; not validated clinical quality classes."}
    return labeled


class MRIPipeline:
    """Reusable product module: enhancement + estimated quality + optional true metrics."""
    def __init__(self, enhancer, quality_model, device):
        self.enhancer = enhancer.eval().to(device)
        self.quality_model = quality_model.eval().to(device)
        self.device = device

    @classmethod
    def from_folder(cls, folder, device=None):
        folder = Path(folder)
        cfg = Config(**json.loads((folder / "config.json").read_text()))
        device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        enhancer, quality_model = UNet(cfg.base_channels), QualityNet()
        enhancer.load_state_dict(torch.load(folder / "enhancer_best.pt", map_location="cpu", weights_only=False)["model"])
        quality_model.load_state_dict(torch.load(folder / "quality_best.pt", map_location="cpu", weights_only=False)["model"])
        instance = cls(enhancer, quality_model, device)
        instance.image_size = cfg.image_size
        return instance

    def process(self, normalized_image, reference=None):
        image = np.asarray(normalized_image, dtype=np.float32)
        if image.ndim != 2 or not np.isfinite(image).all() or image.min() < 0 or image.max() > 1:
            raise ValueError("Supply one finite, normalized grayscale image in [0,1].")
        if min(image.shape) < 32 or any(side % 8 for side in image.shape):
            raise ValueError("Image dimensions must be at least 32 and divisible by 8.")
        if hasattr(self, "image_size") and image.shape != (self.image_size, self.image_size):
            raise ValueError(f"This model was trained at {self.image_size}x{self.image_size}; resize the input first.")
        def operation():
            before = estimate_quality(image, self.quality_model, self.device)
            enhanced = enhance_array(image, self.enhancer, self.device)
            after = estimate_quality(enhanced, self.quality_model, self.device)
            return enhanced, before, after
        (enhanced, before, after), elapsed_ms = timed_call(operation, self.device)
        report = {"before_estimated": before, "after_estimated": after, "pipeline_ms": elapsed_ms,
                  "score_scope": "Synthetic Gaussian/Rician degradation of T1 MRI; estimates require external validation."}
        if reference is not None:
            report["before_measured"] = true_metrics(reference, image)
            report["after_measured"] = true_metrics(reference, enhanced)
        return enhanced, label_quality_report(report)


def load_demo_image(file, image_size):
    file = Path(file)
    if file.name.lower().endswith((".nii", ".nii.gz")):
        image = nib.as_closest_canonical(nib.load(file))
        if len(image.shape) != 3:
            raise ValueError("Upload a 3D T1 NIfTI volume.")
        volume = normalize_volume(image.get_fdata(dtype=np.float32))
        array = np.flipud(volume[:, :, volume.shape[2] // 2].T)
    else:
        array = cv2.imread(str(file), cv2.IMREAD_GRAYSCALE)
        if array is None:
            raise ValueError("Use PNG, JPG, BMP, TIFF, .nii or .nii.gz.")
        array = array.astype(np.float32) / 255
    # Keep the full slice shape ratio; square stretching can make anatomy look distorted.
    result = resize_with_padding(array, image_size).astype(np.float32)
    if not np.isfinite(result).all() or float(np.ptp(result)) < 1e-6:
        raise ValueError("The image is empty or constant. Upload a visible brain MRI slice.")
    return result

'''

FINAL_MODEL_PATH = Path('/content/mri_quality_final.py') if Path('/content').is_dir() else Path('mri_quality_final.py')
FINAL_MODEL_PATH.write_text(FINAL_MODEL_SOURCE, encoding='utf-8')
exec(compile(FINAL_MODEL_SOURCE, str(FINAL_MODEL_PATH), 'exec'), globals())
FINAL_STEP_SOURCES = {}
print('Project functions loaded. Continue to settings.')

In [ ]:
#@title 2b. Experiment settings
PROFILE = "standard" #@param ["standard", "quick"]
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    project_root = '/content/drive/MyDrive/MRI_Group06'
else:
    project_root = '/content/mri_group06'
cfg = Config.for_profile(PROFILE, root=project_root)
device = setup(cfg)
print(json.dumps(asdict(cfg), indent=2))
print('Standard: 20 subjects, 48 slices/subject, 128×128 images.')
print('U-Net: up to 25 epochs; QA: 12-epoch warm start, then up to 40 fine-tuning epochs.')

## 3. Download and prepare brain MRI

Download public OpenNeuro T1 MRI. Split **by subject**: 70% development / 30% test,
with validation inside development. Standard uses 11 training, 3 validation and 6 test
subjects. Every resized image is clipped to `[0,1]`. Keep the saved subject manifest.

In [ ]:
records = acquire_dataset(cfg)
data, run_dir = prepare_data(records, cfg)
loaders = make_pair_loaders(data, cfg)
display(pd.DataFrame(records)[['subject', 'size', 'key']].head())
low_quality, reference = next(iter(loaders['train']))
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, image, title in zip(axes, [reference[0, 0], low_quality[0, 0]],
                             ['Reference MRI', 'Simulated low-quality MRI']):
    ax.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
plt.close(fig)

## 3a. Exploratory data analysis (EDA)

Run after Step 3 has prepared the MRI slices. This cell summarizes image-level features and metadata across train, validation, and test. It creates a missingness matrix, numerical-feature KDE distributions, a correlation heatmap, and categorical count plots. These descriptive features are not clinical measurements.


In [ ]:
# EDA: engineered slice statistics + dataset metadata
from scipy.stats import gaussian_kde, entropy
from pathlib import Path

EDA_FEATURES = [
    "foreground_fraction", "foreground_mean", "foreground_std",
    "intensity_entropy", "laplacian_variance", "gradient_mean",
    "high_frequency_residual_median",
]

def _eda_feature_row(image):
    image = np.asarray(image, dtype=np.float32)
    mask = image > 0.08
    roi = image[mask] if mask.any() else image.ravel()
    hist = np.histogram(roi, bins=64, range=(0, 1))[0].astype(np.float64)
    probabilities = hist / max(hist.sum(), 1.0)
    lap = cv2.Laplacian(image, cv2.CV_32F)
    gx = cv2.Sobel(image, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(image, cv2.CV_32F, 0, 1, ksize=3)
    gradient = np.sqrt(gx * gx + gy * gy)
    smooth = cv2.GaussianBlur(image, (3, 3), 0.7)
    high_frequency = np.abs(image - smooth)
    return {
        "foreground_fraction": float(mask.mean()),
        "foreground_mean": float(np.mean(roi)),
        "foreground_std": float(np.std(roi)),
        "intensity_entropy": float(entropy(probabilities[probabilities > 0])),
        "laplacian_variance": float(np.var(lap[mask]) if mask.any() else np.var(lap)),
        "gradient_mean": float(np.mean(gradient[mask]) if mask.any() else np.mean(gradient)),
        # This is a descriptive high-frequency proxy, not calibrated MRI noise/SNR.
        "high_frequency_residual_median": float(np.median(high_frequency[mask]) if mask.any() else np.median(high_frequency)),
    }

_eda_rows = []
for _split in ("train", "validation", "test"):
    _split_data = data[_split]
    for _image, _meta in zip(_split_data["images"], _split_data["metadata"]):
        _eda_rows.append({"split": _split, "subject": _meta["subject"],
                          "slice_index": int(_meta["slice"]), **_eda_feature_row(_image)})
eda_features = pd.DataFrame(_eda_rows)
EDA_DIR = Path(run_dir) / "analysis"
EDA_DIR.mkdir(parents=True, exist_ok=True)
eda_features.to_csv(EDA_DIR / "eda_slice_features.csv", index=False)

# Missingness matrix for the actual metadata + engineered feature table.
_missing = eda_features.isna().to_numpy(dtype=np.uint8)
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(_missing.T, aspect="auto", interpolation="nearest", cmap="Reds", vmin=0, vmax=1)
ax.set_yticks(range(len(eda_features.columns)))
ax.set_yticklabels(eda_features.columns)
ax.set_xlabel("MRI slice row")
ax.set_title(f"EDA missingness matrix ({int(_missing.sum())} missing cells)")
if not _missing.any():
    ax.text(0.5, -0.22, "No missing values detected in the prepared table.",
            transform=ax.transAxes, ha="center", color="#15803d")
fig.tight_layout()
fig.savefig(EDA_DIR / "eda_missingness_matrix.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)

# Correlation heatmap over numerical image features.
_corr = eda_features[EDA_FEATURES].corr(method="pearson")
fig, ax = plt.subplots(figsize=(9, 7))
_im = ax.imshow(_corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(EDA_FEATURES))); ax.set_xticklabels(EDA_FEATURES, rotation=45, ha="right")
ax.set_yticks(range(len(EDA_FEATURES))); ax.set_yticklabels(EDA_FEATURES)
for _i in range(len(EDA_FEATURES)):
    for _j in range(len(EDA_FEATURES)):
        ax.text(_j, _i, f"{_corr.iloc[_i, _j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(_im, ax=ax, label="Pearson correlation")
ax.set_title("MRI slice-feature correlation heatmap")
fig.tight_layout(); fig.savefig(EDA_DIR / "eda_correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)

# Numerical feature distributions with KDE curves.
fig, axes = plt.subplots(3, 3, figsize=(14, 11)); axes = axes.ravel()
for _ax, _feature in zip(axes, EDA_FEATURES):
    _values = eda_features[_feature].dropna().to_numpy(dtype=float)
    _ax.hist(_values, bins=28, density=True, alpha=0.45, color="#4f83cc", label="Histogram")
    if len(_values) > 2 and np.std(_values) > 1e-10:
        _grid = np.linspace(_values.min(), _values.max(), 200)
        _ax.plot(_grid, gaussian_kde(_values)(_grid), color="#b91c1c", lw=2, label="KDE")
    _ax.set_title(_feature.replace("_", " ").title()); _ax.set_ylabel("Density")
    _ax.legend(fontsize=8)
for _ax in axes[len(EDA_FEATURES):]: _ax.axis("off")
fig.suptitle("Numerical MRI feature distributions", y=1.01)
fig.tight_layout(); fig.savefig(EDA_DIR / "eda_feature_distributions_kde.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)

# Count plots for categorical dataset fields.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
eda_features["split"].value_counts().reindex(["train", "validation", "test"]).plot.bar(
    ax=axes[0], color=["#4f83cc", "#e6a23c", "#55a868"])
axes[0].set_title("Slice counts by split"); axes[0].set_xlabel("Split"); axes[0].set_ylabel("Slices")
eda_features["subject"].value_counts().sort_index().plot.bar(ax=axes[1], color="#8172b2")
axes[1].set_title("Slice counts by subject"); axes[1].set_xlabel("Subject"); axes[1].set_ylabel("Slices")
axes[1].tick_params(axis="x", rotation=75)
fig.tight_layout(); fig.savefig(EDA_DIR / "eda_categorical_countplots.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)
print("EDA table and plots saved to:", EDA_DIR)
print("Feature note: high-frequency residual and Laplacian variance are image statistics, not clinical noise/SNR measurements.")


## 4. Train the enhancement U-Net

Residual U-Net; standard profile uses 256 × 256 slices and a small batch for Colab GPU memory. Training combines L1, SSIM, and edge losses to reduce over-smoothing. The quick profile stays at 128 × 128. Validation loss controls early stopping and checkpoint selection; the new preprocessing/loss signature prevents resuming incompatible older enhancer checkpoints.


In [ ]:
enhancer = UNet(cfg.base_channels)
print(f"U-Net parameters: {sum(p.numel() for p in enhancer.parameters()):,}")
enhancer, enhancement_history = fit_model(
    enhancer, loaders, device, cfg, run_dir, "enhancer", cfg.epochs, enhancement_loss
)
plot_history(enhancement_history, "U-Net training and validation", run_dir / "enhancer_training.png")

## 5. Train the baseline quality CNN

This is the **warm start used by the improved model**, not a second final model.
It predicts PSNR/SSIM proxies without a reference. Keep this stage to preserve
the two-stage QA training method from your supplied notebook.

In [ ]:
qa_loaders = make_qa_loaders(data, enhancer, device, cfg)
quality_model, quality_history = fit_model(
    QualityNet(), qa_loaders, device, cfg, run_dir, "quality", cfg.qa_epochs, nn.MSELoss()
)
plot_history(quality_history, "Quality CNN training and validation", run_dir / "quality_training.png")
# Release the prepared quality-training tensors before evaluation.
del qa_loaders

## 6. Improve the quality model

**6a:** prepare three corruption variants per slice, including near-clean images,
plus frozen-U-Net outputs. Balance training sampling only; validation stays fixed.
Labels come from measured reference SSIM. Keep the subject split unchanged.

**6b:** fine-tune from Step 5 with Adam (`3e-4`), learning-rate scheduling and early
stopping, up to 40 epochs. Select validation balanced accuracy, with MSE breaking ties;
retain original QA weights if no candidate wins. The U-Net stays fixed. No test tuning.

In [ ]:
#@title 6a. Prepare varied, balanced QA data
_stage_source = r'''
# Step 6a: prepare a separate quality-model improvement run.
# Run after the Steps 1-5 in the SAME Colab session.
from pathlib import Path
import hashlib
import json
import numpy as np

missing = [name for name in ("data", "cfg", "run_dir") if name not in globals()]
if missing:
    raise RuntimeError("Run Steps 1-5 first. Missing: " + ", ".join(missing))

QA_SOURCE_RUN_DIR = Path(run_dir).resolve()
required_files = ("config.json", "subject_manifest.json", "enhancer_best.pt", "quality_best.pt")
for name in required_files:
    if not (QA_SOURCE_RUN_DIR / name).is_file():
        raise FileNotFoundError(f"Missing trained-run file: {QA_SOURCE_RUN_DIR / name}")

QA_SOURCE_CONFIG = json.loads((QA_SOURCE_RUN_DIR / "config.json").read_text())
QA_SOURCE_MANIFEST = json.loads((QA_SOURCE_RUN_DIR / "subject_manifest.json").read_text())
for name in ("image_size", "base_channels", "seed", "n_subjects", "slices_per_subject"):
    if getattr(cfg, name) != QA_SOURCE_CONFIG[name]:
        raise ValueError(f"cfg.{name} differs from the saved run. Restore its original value.")

subject_ids = {
    split: {row["subject"] for row in QA_SOURCE_MANIFEST[split]}
    for split in ("train", "validation", "test")
}
if any(not ids for ids in subject_ids.values()) or any(
    subject_ids[a] & subject_ids[b]
    for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))
):
    raise ValueError("The saved subject splits are empty or overlap.")

for split in ("train", "validation"):
    images, metadata = data[split]["images"], data[split]["metadata"]
    if images.shape != (len(metadata), cfg.image_size, cfg.image_size):
        raise ValueError(f"Unexpected {split} image shape.")
    if {row["subject"] for row in metadata} != subject_ids[split]:
        raise ValueError(f"{split} data does not match the saved subject split.")
    if not np.isfinite(images).all() or images.min() < 0 or images.max() > 1:
        raise ValueError(f"{split} images must be finite and normalized to [0, 1].")

QA_SOURCE_HASHES = {}
for name in required_files:
    digest = hashlib.sha256()
    with (QA_SOURCE_RUN_DIR / name).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    QA_SOURCE_HASHES[name] = digest.hexdigest()

QA_SETTINGS = {
    "epochs": 40,
    "learning_rate": 3e-4,
    "patience": 10,
    "batch_size": 16,
    "variants_per_slice": 3,
    "lr_factor": 0.5,
    "lr_patience": 3,
    "seed": cfg.seed,
}

print("Step 6a ready. Original checkpoints and subject split preserved.")
print("QA settings:", QA_SETTINGS)
print("No training or test evaluation has started.")


# Step 6a: varied quality examples and balanced TRAINING sampling.
# Paste immediately after the preparation code above; this cell does not train a model.
import hashlib
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

needed = ("QA_SETTINGS", "QA_SOURCE_RUN_DIR", "QA_SOURCE_CONFIG",
          "QA_SOURCE_HASHES", "data", "UNet", "degrade", "true_metrics", "enhance_array")
missing = [name for name in needed if name not in globals()]
if missing:
    raise RuntimeError("Run Steps 1-5 and the preparation code above first. Missing: " + ", ".join(missing))
if int(QA_SETTINGS["variants_per_slice"]) < 3:
    raise ValueError("Use at least 3 variants_per_slice to cover all degradation ranges.")

for name, expected in QA_SOURCE_HASHES.items():
    digest = hashlib.sha256()
    with (QA_SOURCE_RUN_DIR / name).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != expected:
        raise RuntimeError(f"Original run file changed: {name}. Rerun the preparation code above.")

QA_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
QA_THRESHOLDS = (0.60, 0.80)
QA_CLASS_NAMES = ("Bad", "Medium", "Good")

# Reload the saved U-Net instead of depending on mutable in-memory weights.
_qa_teacher = UNet(QA_SOURCE_CONFIG["base_channels"])
_qa_teacher.load_state_dict(torch.load(
    QA_SOURCE_RUN_DIR / "enhancer_best.pt", map_location="cpu", weights_only=False
)["model"])
_qa_teacher = _qa_teacher.to(QA_DEVICE).eval().requires_grad_(False)


def _qa_make_arrays(split, seed_offset, teacher):
    references = data[split]["images"]
    variants = int(QA_SETTINGS["variants_per_slice"])
    count = len(references) * variants * 2
    inputs = np.empty((count, 1, *references.shape[1:]), dtype=np.float32)
    targets = np.empty((count, 2), dtype=np.float32)
    classes = np.empty(count, dtype=np.int64)
    row = 0
    for index, clean in enumerate(tqdm(references, desc=f"Prepare QA {split}")):
        for variant in range(variants):
            rng = np.random.default_rng(np.random.SeedSequence(
                [int(QA_SETTINGS["seed"]), seed_offset, index, variant]
            ))
            band = variant % 3
            if band == 0:  # Include near-clean images; labels still use measured SSIM.
                scale, sigma = 1, rng.uniform(0.003, 0.015)
            elif band == 1:
                scale, sigma = int(rng.choice([2, 3])), rng.uniform(0.02, 0.07)
            else:
                scale, sigma = int(rng.choice([3, 4])), rng.uniform(0.07, 0.12)
            degraded = degrade(clean, rng, QA_SOURCE_CONFIG["noise_kind"],
                               scale=scale, sigma=float(sigma))
            enhanced = enhance_array(degraded, teacher, QA_DEVICE)
            for candidate in (degraded, enhanced):
                measured = true_metrics(clean, candidate)
                psnr, ssim = float(measured["psnr_db"]), float(measured["ssim"])
                if not np.isfinite(ssim) or np.isnan(psnr) or psnr == -np.inf:
                    raise ValueError(f"Invalid reference metrics in {split}.")
                if not np.isfinite(candidate).all():
                    raise ValueError(f"Non-finite QA image in {split}.")
                inputs[row, 0] = candidate
                targets[row] = [np.clip(psnr, 0, 60) / 60,
                                (np.clip(ssim, -1, 1) + 1) / 2]
                classes[row] = np.searchsorted(QA_THRESHOLDS, ssim, side="right")
                row += 1
    return inputs, targets, classes


class QAUpgradeDataset(Dataset):
    def __init__(self, inputs, targets, classes, training=False):
        self.inputs, self.targets = torch.from_numpy(inputs), torch.from_numpy(targets)
        self.classes, self.training, self.epoch = classes, training, 0
        self.seed = int(QA_SETTINGS["seed"])

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        image = self.inputs[index]
        if self.training:
            rng = np.random.default_rng(np.random.SeedSequence(
                [self.seed, 731003, self.epoch, int(index)]
            ))
            image = torch.rot90(image, int(rng.integers(4)), dims=(-2, -1))
            if rng.random() < 0.5:
                image = torch.flip(image, dims=(-1,))
        return image.contiguous(), self.targets[index]


try:
    qa_train_data = QAUpgradeDataset(*_qa_make_arrays("train", 710000, _qa_teacher), training=True)
    qa_validation_data = QAUpgradeDataset(*_qa_make_arrays("validation", 720000, _qa_teacher))
finally:
    _qa_teacher.to("cpu")
    del _qa_teacher

QA_CLASS_COUNTS = {}
for split, dataset in (("train", qa_train_data), ("validation", qa_validation_data)):
    counts = np.bincount(dataset.classes, minlength=3)
    QA_CLASS_COUNTS[split] = counts
    print(split, "actual class counts:", dict(zip(QA_CLASS_NAMES, counts.tolist())))
    if (counts == 0).any():
        print("Coverage limited: missing", [QA_CLASS_NAMES[i] for i in np.where(counts == 0)[0]])

# Each present quality class receives equal expected training weight.
weights = 1.0 / QA_CLASS_COUNTS["train"][qa_train_data.classes]
QA_SAMPLER_GENERATOR = torch.Generator().manual_seed(int(QA_SETTINGS["seed"]) + 730000)
QA_LOADER_GENERATOR = torch.Generator().manual_seed(int(QA_SETTINGS["seed"]) + 740000)
qa_sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double),
    num_samples=len(qa_train_data), replacement=True, generator=QA_SAMPLER_GENERATOR)

qa_upgrade_loaders = {
    "train": DataLoader(qa_train_data, batch_size=int(QA_SETTINGS["batch_size"]),
        sampler=qa_sampler, num_workers=0, pin_memory=QA_DEVICE.type == "cuda",
        generator=QA_LOADER_GENERATOR),
    "validation": DataLoader(qa_validation_data, batch_size=int(QA_SETTINGS["batch_size"]),
        shuffle=False, num_workers=0, pin_memory=QA_DEVICE.type == "cuda",
        generator=QA_LOADER_GENERATOR),
}

# Tie future training checkpoints to these exact inputs and source weights.
signature = hashlib.sha256(json.dumps({"version": 1, "settings": QA_SETTINGS,
    "sources": QA_SOURCE_HASHES, "thresholds": QA_THRESHOLDS}, sort_keys=True).encode())
for dataset in (qa_train_data, qa_validation_data):
    for array in (dataset.inputs.numpy(), dataset.targets.numpy(), dataset.classes):
        signature.update(memoryview(np.ascontiguousarray(array)))
QA_DATA_SIGNATURE = signature.hexdigest()
for dataset in (qa_train_data, qa_validation_data):
    dataset.input_signature = QA_DATA_SIGNATURE

print(f"Step 6a ready: {len(qa_train_data):,} training and {len(qa_validation_data):,} validation examples.")
print("Training sampling is balanced in expectation; validation is fixed and unbalanced.")
print("Test images were not used. Continue to Step 6b for fine-tuning.")
'''
FINAL_STEP_SOURCES['quality_prepare.py'] = _stage_source
exec(compile(_stage_source, 'quality_prepare.py', "exec"), globals())

In [ ]:
#@title 6b. Fine-tune and select the QA checkpoint
_stage_source = r'''
# Step 6b: fine-tune QualityNet; select checkpoints using validation only.
from pathlib import Path
import hashlib
import json
import shutil
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display

needed = ("QA_SETTINGS", "QA_SOURCE_RUN_DIR", "QA_SOURCE_CONFIG", "QA_SOURCE_HASHES",
          "QA_DATA_SIGNATURE", "QA_DEVICE", "QA_THRESHOLDS", "QA_CLASS_NAMES",
          "QA_SAMPLER_GENERATOR", "QA_LOADER_GENERATOR", "qa_upgrade_loaders",
          "qa_train_data", "qa_validation_data", "QualityNet")
missing = [name for name in needed if name not in globals()]
if missing:
    raise RuntimeError("Run Step 6a first. Missing: " + ", ".join(missing))
for dataset in (qa_train_data, qa_validation_data):
    if dataset.input_signature != QA_DATA_SIGNATURE:
        raise RuntimeError("Prepared data changed. Rerun Step 6a.")

def _qa_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for name, expected in QA_SOURCE_HASHES.items():
    if _qa_hash(QA_SOURCE_RUN_DIR / name) != expected:
        raise RuntimeError(f"Source file changed: {name}. Rerun Step 6a.")

# Separate folder; never reuse the original optimizer or evaluation CSVs.
qa_run_info = {"version": 1, "input_signature": QA_DATA_SIGNATURE,
    "sources": QA_SOURCE_HASHES, "settings": QA_SETTINGS,
    "selection": "validation balanced accuracy, then MSE (minimum decrease 1e-6)"}
qa_run_id = hashlib.sha256(json.dumps(qa_run_info, sort_keys=True).encode()).hexdigest()[:12]
QA_UPGRADE_DIR = QA_SOURCE_RUN_DIR.parent / f"{QA_SOURCE_RUN_DIR.name}_qa_{qa_run_id}"
QA_UPGRADE_DIR.mkdir(parents=True, exist_ok=True)
info_path = QA_UPGRADE_DIR / "qa_upgrade.json"
if info_path.exists() and json.loads(info_path.read_text()) != qa_run_info:
    raise RuntimeError("Upgrade folder belongs to different training inputs.")
info_path.write_text(json.dumps(qa_run_info, indent=2))
for name in ("config.json", "subject_manifest.json", "enhancer_best.pt"):
    destination = QA_UPGRADE_DIR / name
    if destination.exists():
        if _qa_hash(destination) != QA_SOURCE_HASHES[name]:
            raise RuntimeError(f"Unexpected existing upgrade file: {name}")
    else:
        shutil.copy2(QA_SOURCE_RUN_DIR / name, destination)

def _qa_stats(predictions, targets, classes):
    if predictions.shape != targets.shape or predictions.shape != (len(classes), 2):
        raise ValueError("Validation predictions and labels do not align.")
    if not len(classes) or not np.isfinite(predictions).all():
        raise ValueError("Empty or non-finite validation predictions.")
    labels = np.searchsorted(QA_THRESHOLDS, predictions[:, 1] * 2 - 1, side="right")
    matrix = np.zeros((3, 3), dtype=np.int64)
    np.add.at(matrix, (classes, labels), 1)
    support, predicted = matrix.sum(1), matrix.sum(0)
    recall = np.divide(matrix.diagonal(), support,
                       out=np.zeros(3, dtype=float), where=support > 0)
    f1 = np.divide(2 * matrix.diagonal(), support + predicted,
                   out=np.zeros(3, dtype=float), where=(support + predicted) > 0)
    return {"validation_loss": float(np.mean((predictions - targets) ** 2)),
        "accuracy_pct": float(100 * matrix.trace() / matrix.sum()),
        "balanced_accuracy_pct": float(100 * recall[support > 0].mean()),
        "macro_f1_pct": float(100 * f1.mean()),
        "capped_psnr_mae_db": float(np.mean(np.abs(predictions[:, 0] - targets[:, 0])) * 60),
        "ssim_mae": float(np.mean(np.abs(predictions[:, 1] - targets[:, 1])) * 2)}

@torch.inference_mode()
def _qa_validate(model):
    model.eval()
    outputs, targets = [], []
    for images, labels in qa_upgrade_loaders["validation"]:
        outputs.append(model(images.to(QA_DEVICE)).cpu().numpy())
        targets.append(labels.numpy())
    return _qa_stats(np.concatenate(outputs), np.concatenate(targets), qa_validation_data.classes)

def _qa_is_better(current, previous):
    difference = current["balanced_accuracy_pct"] - previous["balanced_accuracy_pct"]
    return difference > 1e-9 or (abs(difference) <= 1e-9 and
        current["validation_loss"] < previous["validation_loss"] - 1e-6)

def _qa_save(payload, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

torch.manual_seed(int(QA_SETTINGS["seed"]) + 750000)
if QA_DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(int(QA_SETTINGS["seed"]) + 750000)
qa_finetuned_model = QualityNet().to(QA_DEVICE)
qa_finetuned_model.load_state_dict(torch.load(QA_SOURCE_RUN_DIR / "quality_best.pt",
    map_location="cpu", weights_only=False)["model"])
qa_baseline_validation = _qa_validate(qa_finetuned_model)

# A NEW optimizer starts at the lower learning rate.
optimizer = torch.optim.Adam(qa_finetuned_model.parameters(), lr=float(QA_SETTINGS["learning_rate"]))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
    factor=float(QA_SETTINGS["lr_factor"]), patience=int(QA_SETTINGS["lr_patience"]))
loss_fn = torch.nn.MSELoss()
best_path, last_path = QA_UPGRADE_DIR / "quality_best.pt", QA_UPGRADE_DIR / "quality_last.pt"
best, start, stale, history = dict(qa_baseline_validation), 0, 0, []

if last_path.exists():
    state = torch.load(last_path, map_location="cpu", weights_only=False)
    if state["input_signature"] != QA_DATA_SIGNATURE or not best_path.exists():
        raise RuntimeError("Resume checkpoint does not match this prepared dataset.")
    qa_finetuned_model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    start, stale, best, history = state["epoch"] + 1, state["stale"], state["best"], state["history"]
    qa_baseline_validation = state["baseline"]
    torch.set_rng_state(state["torch_rng"].cpu())
    if QA_DEVICE.type == "cuda" and state["cuda_rng"] is not None:
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda_rng"]])
    QA_SAMPLER_GENERATOR.set_state(state["sampler_rng"].cpu())
    QA_LOADER_GENERATOR.set_state(state["loader_rng"].cpu())
    print(f"Resuming this upgrade at epoch {start + 1}.")
else:
    QA_SAMPLER_GENERATOR.manual_seed(int(QA_SETTINGS["seed"]) + 730000)
    QA_LOADER_GENERATOR.manual_seed(int(QA_SETTINGS["seed"]) + 740000)
    _qa_save({"model": qa_finetuned_model.state_dict(), "config": QA_SOURCE_CONFIG,
        "epoch": -1, "selected_from": "original model", "validation": best,
        "input_signature": QA_DATA_SIGNATURE}, best_path)

print("Device:", QA_DEVICE)
print("Original validation balanced accuracy:", round(qa_baseline_validation["balanced_accuracy_pct"], 2), "%")
print("Selection: highest validation balanced accuracy; lower MSE breaks ties.")
for epoch in range(start, int(QA_SETTINGS["epochs"])):
    if stale >= int(QA_SETTINGS["patience"]):
        print("Early stopping reached.")
        break
    started = time.perf_counter()
    qa_train_data.epoch = epoch
    qa_finetuned_model.train()
    total, count = 0.0, 0
    learning_rate = optimizer.param_groups[0]["lr"]
    for images, targets in qa_upgrade_loaders["train"]:
        images, targets = images.to(QA_DEVICE), targets.to(QA_DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(qa_finetuned_model(images), targets)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss; resume from the last completed epoch.")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(qa_finetuned_model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * len(images)
        count += len(images)
    current = _qa_validate(qa_finetuned_model)
    scheduler.step(current["validation_loss"])
    improved = _qa_is_better(current, best)
    if improved:
        best, stale = dict(current), 0
        _qa_save({"model": qa_finetuned_model.state_dict(), "config": QA_SOURCE_CONFIG,
            "epoch": epoch, "selected_from": "fine-tuning", "validation": best,
            "input_signature": QA_DATA_SIGNATURE}, best_path)
    else:
        stale += 1
    if QA_DEVICE.type == "cuda":
        torch.cuda.synchronize()
    history.append({"epoch": epoch + 1, "train_loss": total / count,
        **current, "learning_rate": learning_rate, "selected": improved,
        "seconds": time.perf_counter() - started})
    _qa_save({"model": qa_finetuned_model.state_dict(), "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "epoch": epoch, "stale": stale,
        "best": best, "history": history, "baseline": qa_baseline_validation,
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if QA_DEVICE.type == "cuda" else None,
        "sampler_rng": QA_SAMPLER_GENERATOR.get_state(), "loader_rng": QA_LOADER_GENERATOR.get_state(),
        "input_signature": QA_DATA_SIGNATURE}, last_path)
    pd.DataFrame(history).to_csv(QA_UPGRADE_DIR / "quality_history.csv", index=False)
    print(f"Epoch {epoch+1:02d}/{QA_SETTINGS['epochs']} | train={total/count:.5f} | "
          f"val={current['validation_loss']:.5f} | accuracy={current['accuracy_pct']:.2f}% | "
          f"balanced={current['balanced_accuracy_pct']:.2f}% | lr={learning_rate:.1e}"
          + (" | selected" if improved else ""))

selected_checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
qa_finetuned_model.load_state_dict(selected_checkpoint["model"])
qa_finetuned_model.eval()
qa_selected_validation = _qa_validate(qa_finetuned_model)
qa_upgrade_history = pd.DataFrame(history)
qa_validation_comparison = pd.DataFrame([qa_baseline_validation, qa_selected_validation],
    index=["Original model", "Selected model"])
qa_validation_comparison.to_csv(QA_UPGRADE_DIR / "validation_comparison.csv")
display(qa_validation_comparison.round(4))
print("Selected checkpoint:", selected_checkpoint["selected_from"])
print("Saved to:", QA_UPGRADE_DIR)
print("PSNR MAE above uses capped [0,60] dB targets; regression errors can increase.")
print("Validation balanced accuracy averages classes present; macro-F1 includes all 3.")
print("These are validation results. Test accuracy has not been evaluated by this cell.")
print("Step 6 complete. Step 7 evaluates and activates the selected model.")
'''
FINAL_STEP_SOURCES['quality_finetune.py'] = _stage_source
exec(compile(_stage_source, 'quality_finetune.py', "exec"), globals())

## 7. Evaluate and activate the selected model

One evaluation uses the **selected checkpoint** on preserved test subjects, then activates it for uploads/PDFs. It reports input/enhanced accuracy separately, class support, confusion matrices, PSNR, SSIM, before/after images and processing time. Continue to Step 7a for ROC/AUC curves, regression error analysis, loss curves and an interpretable feature-importance baseline. Image bands: Good ≥0.80 SSIM; Medium ≥0.60; otherwise Bad.

Model-quality project bands use balanced accuracy: Good ≥90%, Medium ≥75%, otherwise Bad. Missing classes or fewer than two subjects gives **Limited evaluation**. Accuracy is model-wide, not confidence in one upload. Do not tune using these scores.


In [ ]:
#@title 7. Evaluate and activate the selected model
_stage_source = r'''
# Step 7: evaluate preserved test subjects, then activate the selected model.
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import torch
from IPython.display import display

needed = ("QA_UPGRADE_DIR", "QA_DATA_SIGNATURE", "QA_DEVICE", "QA_THRESHOLDS",
          "QA_CLASS_NAMES", "QA_SOURCE_HASHES", "data", "Config", "MRIPipeline", "evaluate")
missing = [name for name in needed if name not in globals()]
if missing:
    raise RuntimeError("Run Steps 1-6 first; Step 2 loads all project definitions. "
        ""
        "Missing: " + ", ".join(missing))
if tuple(QA_THRESHOLDS) != (0.60, 0.80):
    raise ValueError("Quality thresholds changed. Keep the Step 6a bands fixed at 0.60/0.80.")

def _qa4_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

_qa4_folder = Path(QA_UPGRADE_DIR)
for name in ("qa_upgrade.json", "config.json", "subject_manifest.json",
             "enhancer_best.pt", "quality_best.pt"):
    if not (_qa4_folder / name).is_file():
        raise FileNotFoundError(f"Missing {name}. Complete Step 6b first.")
_qa4_info = json.loads((_qa4_folder / "qa_upgrade.json").read_text())
if (_qa4_info["input_signature"] != QA_DATA_SIGNATURE
        or _qa4_info["sources"] != QA_SOURCE_HASHES):
    raise RuntimeError("This selected model does not match the prepared training run.")
for name in ("config.json", "subject_manifest.json", "enhancer_best.pt"):
    if _qa4_hash(_qa4_folder / name) != QA_SOURCE_HASHES[name]:
        raise RuntimeError(f"Upgrade run file changed: {name}.")
_qa4_checkpoint = torch.load(_qa4_folder / "quality_best.pt",
                            map_location="cpu", weights_only=False)
if _qa4_checkpoint.get("input_signature") != QA_DATA_SIGNATURE:
    raise RuntimeError("The quality checkpoint belongs to different prepared data.")
_qa4_selected_from = _qa4_checkpoint.get("selected_from", "selected model")
del _qa4_checkpoint

# Verify subject isolation and the test arrays before any inference.
_qa4_manifest = json.loads((_qa4_folder / "subject_manifest.json").read_text())
_qa4_ids = {split: {row["subject"] for row in _qa4_manifest[split]}
            for split in ("train", "validation", "test")}
if any(not ids for ids in _qa4_ids.values()) or any(
    _qa4_ids[a] & _qa4_ids[b]
    for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))
):
    raise ValueError("Subject splits are empty or overlap.")
_qa4_cfg = Config(**json.loads((_qa4_folder / "config.json").read_text()))
for split in ("train", "validation", "test"):
    metadata = data[split]["metadata"]
    if {row["subject"] for row in metadata} != _qa4_ids[split]:
        raise ValueError(f"{split} data does not match the saved subject split.")
    sources = {row["subject"]: row["key"] for row in _qa4_manifest[split]}
    if any(row["source"] != sources[row["subject"]] for row in metadata):
        raise ValueError(f"{split} MRI sources differ from the saved manifest.")
_qa4_images, _qa4_metadata = data["test"]["images"], data["test"]["metadata"]
if _qa4_images.shape != (len(_qa4_metadata), _qa4_cfg.image_size, _qa4_cfg.image_size):
    raise ValueError("Test image shapes do not match the saved configuration.")
if len({(row["subject"], row["slice"]) for row in _qa4_metadata}) != len(_qa4_metadata):
    raise ValueError("Duplicate test subject/slice pairs were found.")
if not np.isfinite(_qa4_images).all() or _qa4_images.min() < 0 or _qa4_images.max() > 1:
    raise ValueError("Test images must be finite and normalized to [0,1].")

def _qa4_grade(balanced, n_classes, n_subjects):
    if n_classes < 3 or n_subjects < 2:
        return "Limited evaluation"
    return "Good" if balanced >= 90 else "Medium" if balanced >= 75 else "Bad"

def _qa4_classification(group):
    required = ("ssim", "estimated_ssim", "psnr_db", "estimated_psnr_db")
    if group.empty or not np.isfinite(group[list(required)].to_numpy(float)).all():
        raise ValueError("Empty or non-finite held-out predictions.")
    true = np.searchsorted(QA_THRESHOLDS, group["ssim"].to_numpy(), side="right")
    predicted = np.searchsorted(QA_THRESHOLDS, group["estimated_ssim"].to_numpy(), side="right")
    matrix = np.zeros((3, 3), dtype=np.int64)
    np.add.at(matrix, (true, predicted), 1)
    support, predicted_support = matrix.sum(1), matrix.sum(0)
    recall = np.divide(matrix.diagonal(), support,
                       out=np.zeros(3, dtype=float), where=support > 0)
    precision = np.divide(matrix.diagonal(), predicted_support,
                          out=np.zeros(3, dtype=float), where=predicted_support > 0)
    f1 = np.divide(2 * matrix.diagonal(), support + predicted_support,
                   out=np.zeros(3, dtype=float), where=(support + predicted_support) > 0)
    balanced = float(100 * recall[support > 0].mean())
    n_classes, n_subjects = int((support > 0).sum()), int(group["subject"].nunique())
    scores = {"accuracy_pct": float(100 * matrix.trace() / matrix.sum()),
        "balanced_accuracy_pct": balanced, "macro_f1_pct": float(100 * f1.mean()),
        "majority_baseline_pct": float(100 * support.max() / support.sum()),
        "psnr_mae_db": float((group["estimated_psnr_db"] - group["psnr_db"]).abs().mean()),
        "ssim_mae": float((group["estimated_ssim"] - group["ssim"]).abs().mean()),
        "n_candidates": int(len(group)), "n_subjects": n_subjects,
        "n_classes_present": n_classes,
        "model_quality": _qa4_grade(balanced, n_classes, n_subjects)}
    per_class = pd.DataFrame({"class": QA_CLASS_NAMES, "support": support,
        "precision_pct": precision * 100, "recall_pct": recall * 100, "f1_pct": f1 * 100})
    confusion = pd.DataFrame(matrix, index=QA_CLASS_NAMES, columns=QA_CLASS_NAMES)
    confusion.index.name, confusion.columns.name = "Actual", "Predicted"
    return scores, per_class, confusion

# Use saved selected weights. Test results cannot change model selection.
_qa4_pipeline = MRIPipeline.from_folder(_qa4_folder, device=QA_DEVICE)
print("Evaluating selected weights:", _qa4_selected_from, flush=True)
print("Fixed held-out subjects:", len(_qa4_ids["test"]), flush=True)
(_qa4_results, _qa4_summary, _qa4_predictions,
 _qa4_regression, _qa4_examples) = evaluate(data, _qa4_pipeline.enhancer,
    _qa4_pipeline.quality_model, QA_DEVICE, _qa4_cfg, _qa4_folder)

QA_TEST_METRICS, _qa4_rows = {}, []
for method, stem in (("Before (degraded)", "input"), ("After (U-Net)", "enhanced")):
    group = _qa4_predictions.loc[_qa4_predictions["method"] == method].copy()
    scores, per_class, confusion = _qa4_classification(group)
    QA_TEST_METRICS[method] = scores
    _qa4_rows.append({"method": method, **scores})
    per_class.to_csv(_qa4_folder / f"qa_{stem}_per_class.csv", index=False)
    confusion.to_csv(_qa4_folder / f"qa_{stem}_confusion.csv")
    print("\n", method)
    display(per_class.round(2))
    display(confusion)

qa_test_summary = pd.DataFrame(_qa4_rows).set_index("method")
qa_test_summary.to_csv(_qa4_folder / "qa_upgrade_test_summary.csv")
display(qa_test_summary.round(4))
display(_qa4_summary.round(4))
(_qa4_folder / "qa_test_provenance.json").write_text(json.dumps({
    "input_signature": QA_DATA_SIGNATURE, "thresholds": QA_THRESHOLDS,
    "test_subjects": sorted(_qa4_ids["test"]), "selected_from": _qa4_selected_from,
    "sha256": {name: _qa4_hash(_qa4_folder / name) for name in (
        "enhancer_best.pt", "quality_best.pt", "subject_manifest.json", "quality_predictions.csv")},
    "metrics": QA_TEST_METRICS,
}, indent=2))

# Activate only after evaluation completes successfully.
run_dir, cfg, device, pipeline = _qa4_folder, _qa4_cfg, QA_DEVICE, _qa4_pipeline
enhancer, quality_model = pipeline.enhancer, pipeline.quality_model
results, summary, quality_predictions = _qa4_results, _qa4_summary, _qa4_predictions
qa_accuracy, examples = _qa4_regression, _qa4_examples
MEDIUM_SSIM, GOOD_SSIM = QA_THRESHOLDS
last_upload_result = None
if isinstance(globals().get("result"), dict) and all(
    key in globals()["result"] for key in ("report", "input", "enhanced", "folder")
):
    globals().pop("result")
for name in ("_report_upload", "uploaded_pdf_path", "report_model_accuracy",
             "report_accuracy_note", "model_accuracy_results"):
    globals().pop(name, None)

primary = QA_TEST_METRICS["Before (degraded)"]
print(f"\nTEST MODEL ACCURACY: {primary['accuracy_pct']:.2f}%")
print(f"BALANCED ACCURACY: {primary['balanced_accuracy_pct']:.2f}%")
print("MODEL QUALITY:", primary["model_quality"])
print("Reference-derived class coverage:", primary["n_classes_present"], "/3")
print("Model quality is a project grade: balanced accuracy >=90 Good, >=75 Medium; else Bad.")
print("A grade requires all 3 classes and at least 2 test subjects.")
print("Balanced accuracy averages supported classes; macro-F1 includes all 3.")
print("This is model-wide synthetic-MRI accuracy, not confidence in an uploaded image.")
print("The U-Net is unchanged; this upgrade targets quality assessment.")
print("Selected model is now active. Results folder:", run_dir)

for function, argument in (("plot_comparisons", examples), ("plot_metric_summary", results),
                           ("plot_quality_accuracy", quality_predictions)):
    if callable(globals().get(function)):
        globals()[function](argument, run_dir)

print("Step 7 complete. Continue to Step 8 for upload, then Step 9 for its PDF.")
print("Do not select checkpoints or adjust thresholds using these test scores.")
print("If earlier scores on these same subjects guided tuning, a final generalization test needs new subjects.")
'''
FINAL_STEP_SOURCES['quality_evaluate.py'] = _stage_source
exec(compile(_stage_source, 'quality_evaluate.py', "exec"), globals())

## 7a. Training, evaluation, and error analysis

Run after Step 7. This adds visual confusion matrices and one-vs-rest ROC/AUC curves for the SSIM-derived quality bands, combined training/validation loss curves, regression residual diagnostics, and an exploratory permutation-importance chart. The feature-importance chart uses hand-crafted MRI slice statistics in a separate Random Forest baseline; it does not explain the U-Net or CNN internals.


In [ ]:
# Post-evaluation analysis: held-out predictions only
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve,
                             mean_absolute_error)

if "quality_predictions" not in globals() or quality_predictions.empty:
    raise RuntimeError("Run Step 7 evaluation first; held-out quality_predictions are missing.")
ANALYSIS_DIR = Path(run_dir) / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# Plot training/validation loss histories for each model stage.
_histories = [
    ("Enhancement U-Net", globals().get("enhancement_history")),
    ("Baseline quality CNN", globals().get("quality_history")),
    ("Quality CNN fine-tuning", globals().get("qa_upgrade_history")),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for _ax, (_title, _history) in zip(axes, _histories):
    if _history is None or len(_history) == 0:
        _ax.text(0.5, 0.5, "No fine-tuning epochs recorded", ha="center", va="center")
        _ax.set_title(_title); _ax.axis("off"); continue
    _df = pd.DataFrame(_history)
    _train_col = "train_loss" if "train_loss" in _df else None
    _val_col = "validation_loss" if "validation_loss" in _df else None
    _x = _df["epoch"] if "epoch" in _df else np.arange(1, len(_df) + 1)
    if _train_col: _ax.plot(_x, _df[_train_col], label="Training", marker="o", ms=3)
    if _val_col: _ax.plot(_x, _df[_val_col], label="Validation", marker="o", ms=3)
    _ax.set_title(_title); _ax.set_xlabel("Epoch"); _ax.set_ylabel("Loss")
    _ax.grid(alpha=0.25); _ax.legend()
fig.tight_layout(); fig.savefig(ANALYSIS_DIR / "training_validation_loss_curves.png", dpi=160)
plt.show(); plt.close(fig)

# Confusion matrices and ROC/AUC using measured test SSIM bands as targets.
_CLASS_NAMES = ["Bad", "Medium", "Good"]
_THRESHOLDS = np.asarray(QA_THRESHOLDS, dtype=float)
_METHODS = [("Before (degraded)", "Input"), ("After (U-Net)", "Enhanced")]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for _row, (_method, _display) in enumerate(_METHODS):
    _group = quality_predictions.loc[quality_predictions["method"] == _method].copy()
    if _group.empty:
        axes[_row, 0].axis("off"); axes[_row, 1].axis("off"); continue
    _actual = np.searchsorted(_THRESHOLDS, _group["ssim"].to_numpy(float), side="right")
    _estimate = _group["estimated_ssim"].to_numpy(float)
    _predicted = np.searchsorted(_THRESHOLDS, _estimate, side="right")
    _cm = confusion_matrix(_actual, _predicted, labels=[0, 1, 2])
    _ax = axes[_row, 0]
    _im = _ax.imshow(_cm, cmap="Blues")
    _ax.set_xticks(range(3), _CLASS_NAMES); _ax.set_yticks(range(3), _CLASS_NAMES)
    _ax.set_xlabel("Predicted class"); _ax.set_ylabel("Actual class")
    _ax.set_title(f"{_display} quality confusion matrix")
    for _i in range(3):
        for _j in range(3): _ax.text(_j, _i, str(_cm[_i, _j]), ha="center", va="center")
    fig.colorbar(_im, ax=_ax, fraction=0.046)

    _ax = axes[_row, 1]
    _auc_lines = 0
    for _class_id, _class_name in enumerate(_CLASS_NAMES):
        _binary = (_actual == _class_id).astype(int)
        if len(np.unique(_binary)) < 2:
            continue
        # Ordinal regression score converted to a ranking score for each SSIM band.
        if _class_id == 0: _score = -_estimate
        elif _class_id == 1: _score = -np.abs(_estimate - float(np.mean(_THRESHOLDS)))
        else: _score = _estimate
        _fpr, _tpr, _ = roc_curve(_binary, _score)
        _auc = roc_auc_score(_binary, _score)
        _ax.plot(_fpr, _tpr, lw=2, label=f"{_class_name} vs rest (AUC={_auc:.3f})")
        _auc_lines += 1
    _ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
    _ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel="False positive rate", ylabel="True positive rate",
            title=f"{_display} one-vs-rest ROC")
    _ax.grid(alpha=0.25); _ax.legend(fontsize=8)
    if not _auc_lines: _ax.text(0.5, 0.5, "AUC unavailable: test labels contain one class", ha="center")
fig.tight_layout(); fig.savefig(ANALYSIS_DIR / "confusion_matrices_roc_auc.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)

# Separate actual-vs-predicted, residual, and error-distribution plots by metric.
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for _method, _label in _METHODS:
    _group = quality_predictions.loc[quality_predictions["method"] == _method]
    if _group.empty: continue
    _color = "#4f83cc" if _label == "Input" else "#dd8452"
    # SSIM identity plot (each metric gets its own scale).
    _actual_ssim = _group["ssim"].to_numpy(float)
    _pred_ssim = _group["estimated_ssim"].to_numpy(float)
    axes[0, 0].scatter(_actual_ssim, _pred_ssim, s=12, alpha=0.35, label=_label, color=_color)
    # PSNR identity plot.
    _actual_psnr = _group["psnr_db"].to_numpy(float)
    _pred_psnr = _group["estimated_psnr_db"].to_numpy(float)
    axes[0, 1].scatter(_actual_psnr, _pred_psnr, s=12, alpha=0.35, label=_label, color=_color)
    # Residuals (estimate minus measured) against measured value.
    axes[1, 0].scatter(_actual_ssim, _pred_ssim - _actual_ssim, s=12, alpha=0.35, label=_label, color=_color)
    axes[1, 1].scatter(_actual_psnr, _pred_psnr - _actual_psnr, s=12, alpha=0.35, label=_label, color=_color)
    axes[1, 2].hist(_pred_ssim - _actual_ssim, bins=30, alpha=0.45, density=True, label=_label, color=_color)
    axes[0, 2].hist(_pred_psnr - _actual_psnr, bins=30, alpha=0.45, density=True, label=_label, color=_color)

for _ax, _actual, _pred, _title, _label in (
    (axes[0, 0], quality_predictions["ssim"].to_numpy(float), quality_predictions["estimated_ssim"].to_numpy(float), "SSIM", "SSIM"),
    (axes[0, 1], quality_predictions["psnr_db"].to_numpy(float), quality_predictions["estimated_psnr_db"].to_numpy(float), "PSNR (dB)", "PSNR (dB)"),
):
    _lo, _hi = min(_actual.min(), _pred.min()), max(_actual.max(), _pred.max())
    _ax.plot([_lo, _hi], [_lo, _hi], "k--", label="Identity: y = x")
    _ax.set(xlim=(_lo, _hi), ylim=(_lo, _hi), xlabel=f"Measured {_label}", ylabel=f"Estimated {_label}",
            title=f"Actual vs. predicted {_title}")
    _ax.grid(alpha=0.25); _ax.legend(fontsize=8)
axes[0, 2].set(xlabel="Residual (estimated − measured), dB", ylabel="Density", title="PSNR error distribution")
axes[1, 0].axhline(0, color="k", ls="--", lw=1)
axes[1, 0].set(xlabel="Measured SSIM", ylabel="Residual (estimated − measured)", title="SSIM residual plot")
axes[1, 1].axhline(0, color="k", ls="--", lw=1)
axes[1, 1].set(xlabel="Measured PSNR (dB)", ylabel="Residual (estimated − measured), dB", title="PSNR residual plot")
axes[1, 2].axvline(0, color="k", ls="--", lw=1)
axes[1, 2].set(xlabel="Residual (estimated − measured)", ylabel="Density", title="SSIM error distribution")
for _ax in (axes[0, 2], axes[1, 0], axes[1, 1], axes[1, 2]):
    _ax.grid(alpha=0.25); _ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(ANALYSIS_DIR / "quality_residual_diagnostics.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)

# Separate, interpretable baseline: predict reference SSIM from hand-crafted features.
# Train and test are grouped by subjects already; use only training subjects to fit RF.
def _analysis_quality_examples(split_name, max_slices_per_subject=20, seed_offset=800000):
    _images = data[split_name]["images"]
    _metadata = data[split_name]["metadata"]
    _subjects = sorted({row["subject"] for row in _metadata})
    _selected = []
    for _subject in _subjects:
        _indices = [i for i, row in enumerate(_metadata) if row["subject"] == _subject]
        if len(_indices) > max_slices_per_subject:
            _positions = np.linspace(0, len(_indices) - 1, max_slices_per_subject).round().astype(int)
            _indices = [_indices[j] for j in _positions]
        _selected.extend(_indices)
    _kind_list = ("gaussian", "rician") if cfg.noise_kind == "mixed" else (cfg.noise_kind,)
    _rows, _targets = [], []
    for _kind_index, _kind in enumerate(_kind_list):
        for _severity_index, (_severity, (_scale, _sigma)) in enumerate(SCENARIOS.items()):
            for _index in _selected:
                _clean = _images[_index]
                _rng = np.random.default_rng(cfg.seed + seed_offset + _kind_index*100000 + _severity_index*10000 + _index)
                _candidate = degrade(_clean, _rng, _kind, _scale, _sigma)
                _rows.append(_eda_feature_row(_candidate))
                _targets.append(true_metrics(_clean, _candidate)["ssim"])
    return pd.DataFrame(_rows, columns=EDA_FEATURES), np.asarray(_targets, dtype=float)

_X_train_fi, _y_train_fi = _analysis_quality_examples("train", seed_offset=810000)
_X_test_fi, _y_test_fi = _analysis_quality_examples("test", seed_offset=910000)
_fi_model = RandomForestRegressor(n_estimators=180, min_samples_leaf=3,
                                  random_state=cfg.seed, n_jobs=-1)
_fi_model.fit(_X_train_fi, _y_train_fi)
_fi = permutation_importance(_fi_model, _X_test_fi, _y_test_fi,
                             scoring="neg_mean_absolute_error", n_repeats=5,
                             random_state=cfg.seed, n_jobs=-1)
_fi_table = pd.DataFrame({"feature": EDA_FEATURES,
    "importance_mean": _fi.importances_mean,
    "importance_std": _fi.importances_std}).sort_values("importance_mean")
_fi_table.to_csv(ANALYSIS_DIR / "engineered_feature_importance.csv", index=False)
_fi_mae = mean_absolute_error(_y_test_fi, _fi_model.predict(_X_test_fi))
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(_fi_table["feature"], _fi_table["importance_mean"],
        xerr=_fi_table["importance_std"], color="#4f83cc", alpha=0.9)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Decrease in negative MAE when shuffled (higher = more useful)")
ax.set_title(f"Permutation feature importance: engineered-feature RF baseline\nHeld-out MAE={_fi_mae:.3f} SSIM")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout(); fig.savefig(ANALYSIS_DIR / "engineered_feature_importance.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)
print("Analysis plots and tables saved to:", ANALYSIS_DIR)
print(f"Exploratory RF baseline held-out SSIM MAE: {_fi_mae:.4f}; this is not the CNN's accuracy.")
print("ROC/AUC uses SSIM-derived project bands and continuous estimated SSIM ranking scores, not calibrated class probabilities.")


## 8. Upload an MRI and view its result

Run and click **Choose files**. Select PNG/JPG/BMP/TIFF or `.nii`/`.nii.gz` (up to 128 MB).
NIfTI uses its middle axial slice. See **Good/Medium/Bad**, enhanced images, estimated
scores and processing time. Rerun for another image. This model assesses brain MRI.

In [ ]:
#@title 8. Choose an MRI and analyze it
_stage_source = r'''
# MRI_UPLOAD_UI — paste this entire file into one cell in your existing Colab notebook.
# Run after training/loading the models. The native Choose files uploader opens in this cell.
import html
import json
import math
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# Editable project bands. These thresholds are not radiologist quality labels.
MEDIUM_SSIM = 0.60
GOOD_SSIM = 0.80
MAX_UPLOAD_BYTES = 128 * 1024 * 1024
SUPPORTED_UPLOAD_SUFFIXES = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".nii", ".nii.gz")






def unpack_uploaded_file(value):
    """Accept ipywidgets 7/8 and native Colab upload values; return a safe basename."""
    if not value:
        return None
    if isinstance(value, dict):
        fallback_name, item = next(iter(value.items()))
        if isinstance(item, (bytes, bytearray, memoryview)):
            name, content = fallback_name, item
        else:
            name = item.get("name") or item.get("metadata", {}).get("name") or fallback_name
            content = item["content"]
    else:
        item = value[0]
        name, content = item["name"], item["content"]
    name = str(name).replace("\\", "/").rsplit("/", 1)[-1]
    if not name or not name.lower().endswith(SUPPORTED_UPLOAD_SUFFIXES):
        raise ValueError("Upload a brain MRI as PNG, JPG, BMP, TIFF, .nii or .nii.gz.")
    if not len(content):
        raise ValueError("The uploaded file is empty. Select a readable image.")
    if len(content) > MAX_UPLOAD_BYTES:
        raise ValueError("The file exceeds 128 MB. Upload one slice or a smaller NIfTI volume.")
    return name, bytes(content)


def get_upload_pipeline():
    """Reuse this runtime's model, or reload its saved checkpoints."""
    model = globals().get("pipeline")
    if model is not None and callable(getattr(model, "process", None)):
        return model
    factory, folder = globals().get("MRIPipeline"), globals().get("run_dir")
    if factory is not None and folder is not None:
        folder = Path(folder)
        if all((folder / name).is_file() for name in ("config.json", "enhancer_best.pt", "quality_best.pt")):
            model = factory.from_folder(folder, device=globals().get("device"))
            globals()["pipeline"] = model
            return model
    raise RuntimeError(
        "Models are not loaded. Finish training in Steps 4-7, then run Step 7. "
        "If you already trained in this session, run Step 7 to load the saved checkpoints; no retraining is needed."
    )


def analyze_uploaded_file(value):
    """Run the actual trained pipeline; never invent a label when the model is missing."""
    item = unpack_uploaded_file(value)
    if item is None:
        return None
    filename, content = item
    model = get_upload_pipeline()
    loader = globals().get("load_demo_image")
    if not callable(loader):
        raise RuntimeError("Run the inference definitions in Step 7, then retry this upload.")
    size = getattr(model, "image_size", None) or getattr(globals().get("cfg"), "image_size", None)
    if size is None:
        raise RuntimeError("Model input size is unavailable. Rerun Step 7 to load the trained pipeline.")
    folder = Path(globals().get("run_dir", "/content/mri_upload_results")) / "user_uploads" / str(time.time_ns())
    folder.mkdir(parents=True, exist_ok=True)
    source = folder / filename
    source.write_bytes(content)
    image = loader(source, int(size))
    # Also validate old notebook loaders, which did not reject constant images.
    if float(image.max() - image.min()) < 1e-6:
        raise ValueError("The image is empty or constant. Upload a visible brain MRI slice.")
    enhanced, report = model.process(image)
    report = label_quality_report(report, MEDIUM_SSIM, GOOD_SSIM)
    report["uploaded_filename"] = filename
    report["assessed_resolution"] = [int(size), int(size)]
    image_path, report_path = folder / "enhanced.png", folder / "quality_report.json"
    if not cv2.imwrite(str(image_path), (enhanced.clip(0, 1) * 255).round().astype("uint8")):
        raise IOError("Could not save the enhanced image.")
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return {"input": image, "enhanced": enhanced, "report": report, "image_path": image_path,
            "report_path": report_path, "folder": folder}


def show_upload_result(result):
    if result is None:
        return
    report = result["report"]
    print("UPLOADED IMAGE QUALITY:", report["before_estimated"]["quality_label"], flush=True)
    print("ENHANCED IMAGE QUALITY:", report["after_estimated"]["quality_label"], flush=True)
    print(f"Estimated input SSIM: {report['before_estimated']['estimated_ssim']:.3f} | "
          f"Estimated input PSNR: {report['before_estimated']['estimated_psnr_db']:.2f} dB", flush=True)
    colors = {"Good": "#15803d", "Medium": "#a16207", "Bad": "#b91c1c"}
    cards = []
    for key, title in (("before_estimated", "Uploaded image"), ("after_estimated", "After enhancement")):
        score = report[key]
        label = score["quality_label"]
        cards.append(
            f'<div style="flex:1;border:1px solid #cbd5e1;border-radius:10px;padding:16px">'
            f'<div>{title} · estimated quality</div>'
            f'<div style="font-size:32px;font-weight:700;color:{colors[label]}">{label}</div>'
            f'<div>Estimated SSIM: <b>{score["estimated_ssim"]:.3f}</b></div>'
            f'<div>Estimated PSNR: <b>{score["estimated_psnr_db"]:.2f} dB</b></div></div>'
        )
    display(HTML(f'<h3>{html.escape(report["uploaded_filename"])}</h3>'
                 + '<div style="display:flex;gap:12px;flex-wrap:wrap">' + ''.join(cards) + '</div>'
                 + f'<p>Processing time: <b>{report["pipeline_ms"]:.1f} ms</b> · assessed at '
                 + f'{report["assessed_resolution"][0]} × {report["assessed_resolution"][1]} pixels.</p>'))
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for ax, array, title, key in ((axes[0], result["input"], "Uploaded MRI", "before_estimated"),
                                 (axes[1], result["enhanced"], "Enhanced MRI", "after_estimated")):
        ax.imshow(array, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f'{title}\nEstimated quality: {report[key]["quality_label"]}')
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(result["folder"] / "comparison.png", dpi=150)
    plt.show()
    plt.close(fig)
    print("Labels use project thresholds on predicted SSIM. These are estimates for brain MRI, not general-photo quality ratings.")
    print("True PSNR/SSIM require an aligned reference scan; no reference was uploaded.")
    print("Saved enhanced PNG:", result["image_path"])
    print("Saved quality report:", result["report_path"])
    print("To download, run in a new cell:")
    print("from google.colab import files")
    print("files.download(str(last_upload_result['image_path']))")
    print("# For the report: files.download(str(last_upload_result['report_path']))")


def upload_mri_native():
    """Choose files, run inference, and display output in this executing code cell."""
    from google.colab import files
    global last_upload_result
    last_upload_result = None
    print("1. Click Choose files below and select your MRI image.", flush=True)
    uploaded = files.upload()
    if not uploaded:
        print("No file selected. Run this cell again to upload an MRI.", flush=True)
        return None
    for filename, content in uploaded.items():
        print(f"2. Received {filename}. Running image analysis…", flush=True)
        try:
            result = analyze_uploaded_file({filename: content})
            show_upload_result(result)
            last_upload_result = result
        except Exception as error:
            print(f"Analysis failed for {filename}: {error}", flush=True)
            raise
    return last_upload_result


def show_mri_upload_panel():
    """Compatibility entry point; the native uploader is now the default."""
    return upload_mri_native()


try:
    from google.colab import files as _colab_files
except ImportError:
    print("Upload helpers loaded. Run this cell in Google Colab to open Choose files.")
else:
    # This runs in the code cell itself, without a widget observer or button callback.
    last_upload_result = upload_mri_native()
'''
FINAL_STEP_SOURCES['upload_mri.py'] = _stage_source
exec(compile(_stage_source, 'upload_mri.py', "exec"), globals())

## 9. Generate the uploaded-image PDF report

Run after Step 8 displays results. Download a PDF with both images, their labels,
estimated PSNR/SSIM, timing, **test accuracy percentage, balanced accuracy and model
quality**. Test accuracy comes from Step 7, not from the uploaded image alone.

In [ ]:
#@title 9. Generate and download the PDF report
_stage_source = r'''
# MRI_REPORT_CELL - paste this entire file into a new Colab cell after uploading an MRI.
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
import importlib.util
import csv
import json
import math
import subprocess
import sys
from xml.sax.saxutils import escape


def _report_label(score, medium=0.60, good=0.80):
    if not 0 <= medium < good <= 1:
        raise ValueError("Invalid report thresholds: require 0 <= medium < good <= 1.")
    score = float(score)
    if not math.isfinite(score) or not -1 <= score <= 1:
        raise ValueError("Report SSIM must be finite and in [-1, 1].")
    return "Good" if score >= good else "Medium" if score >= medium else "Bad"


def model_performance_grade(balanced_accuracy_pct, n_classes, n_subjects):
    """Editable project convention, not a clinically validated model rating."""
    score = float(balanced_accuracy_pct)
    if not math.isfinite(score) or not 0 <= score <= 100:
        raise ValueError("Balanced accuracy must be finite and between 0 and 100 percent.")
    if int(n_classes) < 3 or int(n_subjects) < 2:
        return "Limited evaluation"
    return "Good" if score >= 90 else "Medium" if score >= 75 else "Bad"


def get_report_model_accuracy(folder, medium=0.60, good=0.80):
    """Recalculate the held-out input-quality accuracy from saved reference/predicted SSIM."""
    _report_label(0, medium, good)  # Validate the frozen image-quality thresholds.
    folder = Path(folder)
    required = ("quality_predictions.csv", "subject_manifest.json", "enhancer_best.pt", "quality_best.pt")
    missing = [name for name in required if not (folder / name).is_file()]
    if missing:
        raise ValueError("Run Step 7 after training/loading both models; missing " + ", ".join(missing) + ".")
    predictions_path = folder / "quality_predictions.csv"
    for checkpoint in ("enhancer_best.pt", "quality_best.pt"):
        if predictions_path.stat().st_mtime_ns < (folder / checkpoint).stat().st_mtime_ns:
            raise ValueError("Saved predictions are older than " + checkpoint + ". Rerun Step 7, then this report cell.")
    try:
        manifest = json.loads((folder / "subject_manifest.json").read_text(encoding="utf-8"))
    except (OSError, ValueError) as error:
        raise ValueError("Cannot read the held-out subject manifest. Rerun data preparation and Step 7.") from error
    if not isinstance(manifest, dict):
        raise ValueError("The subject manifest must define train, validation and test subjects.")
    subjects = {}
    for split in ("train", "validation", "test"):
        entries = manifest.get(split)
        if not isinstance(entries, list) or not entries:
            raise ValueError("The subject manifest has no valid " + split + " split.")
        ids = [row.get("subject") if isinstance(row, dict) else None for row in entries]
        if any(not isinstance(value, str) or not value.strip() for value in ids):
            raise ValueError("The subject manifest contains invalid subject IDs.")
        if len(set(ids)) != len(ids):
            raise ValueError("Repeated subject IDs in the subject manifest.")
        subjects[split] = set(ids)
    if (subjects["train"] & subjects["validation"] or subjects["train"] & subjects["test"]
            or subjects["validation"] & subjects["test"]):
        raise ValueError("Subject leakage was found between train, validation and test splits. Correct the experiment before reporting accuracy.")
    with predictions_path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not {"subject", "method", "ssim", "estimated_ssim"}.issubset(reader.fieldnames or ()):
            raise ValueError("The held-out predictions lack required columns. Rerun Step 7.")
        all_rows = list(reader)
    if any(row["subject"] not in subjects["test"] for row in all_rows):
        raise ValueError("Prediction rows include subjects outside the held-out test split. Rerun Step 7 on test subjects only.")
    rows = [row for row in all_rows if row["method"] == "Before (degraded)"]
    if not rows:
        raise ValueError("No held-out input predictions were found. Rerun Step 7.")
    classes = ("Bad", "Medium", "Good")
    support, correct = dict.fromkeys(classes, 0), dict.fromkeys(classes, 0)
    identities = set()
    for row in rows:
        actual, predicted = _report_label(row["ssim"], medium, good), _report_label(row["estimated_ssim"], medium, good)
        support[actual] += 1
        correct[actual] += int(actual == predicted)
        if all(row.get(key) not in (None, "") for key in ("slice", "noise", "severity")):
            identity = tuple(row[key] for key in ("subject", "slice", "noise", "severity"))
            if identity in identities:
                raise ValueError("Duplicate test candidates were found. Rerun Step 7 without combining evaluations.")
            identities.add(identity)
    supported = [name for name in classes if support[name]]
    accuracy = 100 * sum(correct.values()) / len(rows)
    balanced = 100 * sum(correct[name] / support[name] for name in supported) / len(supported)
    n_subjects = len({row["subject"] for row in rows})
    return {"accuracy_pct": accuracy, "balanced_accuracy_pct": balanced,
            "n_candidates": len(rows), "n_subjects": n_subjects, "n_classes_present": len(supported),
            "class_support": support, "majority_baseline_pct": 100 * max(support.values()) / len(rows),
            "model_quality": model_performance_grade(balanced, len(supported), n_subjects),
            "medium": medium, "good": good, "method": "Before (degraded)",
            "source": str(predictions_path),
            "scope": "Held-out reference-SSIM-derived quality classes; not the confidence of this uploaded image."}


def _pdf_image_buffer(image):
    import numpy as np
    from PIL import Image as PILImage
    array = np.asarray(image, dtype=np.float32)
    if array.ndim != 2 or not array.size or not np.isfinite(array).all():
        raise ValueError("Report images must be finite, nonempty 2D grayscale arrays.")
    if array.min() < 0 or array.max() > 1:
        raise ValueError("Report images must be normalized to [0,1].")
    buffer = BytesIO()
    PILImage.fromarray((array * 255).round().astype(np.uint8)).save(buffer, format="PNG")
    buffer.seek(0)
    return buffer


def create_uploaded_mri_pdf(upload_result, output_path=None, generated_at=None, demo=False,
                            model_accuracy=None, accuracy_note=None):
    """Create a report from the actual uploaded-image result; performs no new inference."""
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER, TA_LEFT
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.platypus import Image, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle, KeepTogether

    if not isinstance(upload_result, dict) or not all(key in upload_result for key in ("report", "input", "enhanced", "folder")):
        raise ValueError("Upload and analyze an MRI first, then run this report cell.")
    report = upload_result["report"]
    bands = report.get("quality_bands", {})
    medium, good = float(bands.get("medium_at_or_above", 0.60)), float(bands.get("good_at_or_above", 0.80))
    estimates = {}
    for name in ("before_estimated", "after_estimated"):
        entry = dict(report[name])
        label = _report_label(entry["estimated_ssim"], medium, good)
        if entry.get("quality_label", label) != label:
            raise ValueError("Stored quality label does not match this report's SSIM thresholds. Rerun image analysis.")
        psnr = float(entry["estimated_psnr_db"])
        if not math.isfinite(psnr) or not 0 <= psnr <= 60:
            raise ValueError("Estimated report PSNR must be finite and in the model's 0-60 dB range.")
        estimates[name] = {**entry, "quality_label": label, "estimated_psnr_db": psnr}
    milliseconds = float(report["pipeline_ms"])
    if not math.isfinite(milliseconds) or milliseconds < 0:
        raise ValueError("Report processing time must be finite and nonnegative.")
    filename = str(report.get("uploaded_filename", "Uploaded MRI"))
    output_path = Path(output_path or (Path(upload_result["folder"]) / "uploaded_mri_report.pdf"))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    stamp = generated_at or datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    navy, slate, blue = colors.HexColor("#172D45"), colors.HexColor("#526276"), colors.HexColor("#2463A2")
    pale, line = colors.HexColor("#F1F5F9"), colors.HexColor("#D6DFE8")
    label_colors = {"Good": "#15803D", "Medium": "#A16207", "Bad": "#B91C1C"}
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="MRIKicker", fontName="Helvetica-Bold", fontSize=9, leading=12, textColor=blue, spaceAfter=7))
    styles.add(ParagraphStyle(name="MRITitle", fontName="Helvetica-Bold", fontSize=24, leading=28, textColor=navy, spaceAfter=8))
    styles.add(ParagraphStyle(name="MRIFilename", fontName="Helvetica", fontSize=9, leading=12, textColor=slate, wordWrap="CJK", spaceAfter=5))
    styles.add(ParagraphStyle(name="MRIBody", fontName="Helvetica", fontSize=9, leading=13, textColor=navy, spaceAfter=7))
    styles.add(ParagraphStyle(name="MRISmall", fontName="Helvetica", fontSize=8, leading=11, textColor=slate, spaceAfter=5))
    styles.add(ParagraphStyle(name="MRIHeading", fontName="Helvetica-Bold", fontSize=12, leading=16, textColor=navy, spaceBefore=9, spaceAfter=7))
    styles.add(ParagraphStyle(name="MRICenter", fontName="Helvetica-Bold", fontSize=9, leading=12, textColor=navy, alignment=TA_CENTER))
    styles.add(ParagraphStyle(name="MRIGrade", fontName="Helvetica-Bold", fontSize=16, leading=20, alignment=TA_CENTER))
    p = lambda text, style="MRIBody": Paragraph(text, styles[style])
    before, after = estimates["before_estimated"], estimates["after_estimated"]
    width = A4[0] - 84
    story = [p("GROUP 06 / MRI IMAGE QUALITY ASSESSMENT", "MRIKicker"),
             p("Uploaded MRI Report", "MRITitle"),
             p("File: " + escape(filename), "MRIFilename"),
             p("Generated: " + escape(str(stamp)), "MRISmall")]
    if demo:
        story.append(p("DEMONSTRATION ONLY - synthetic image and illustrative scores; not a model evaluation.", "MRIBody"))
    story.append(Spacer(1, 7))
    grade = lambda entry: p('<font color="' + label_colors[entry["quality_label"]] + '">' + entry["quality_label"] + '</font>', "MRIGrade")
    overview = Table([
        [p("Result", "MRICenter"), p("Uploaded image", "MRICenter"), p("After enhancement", "MRICenter")],
        [p("Estimated quality"), grade(before), grade(after)],
        [p("Estimated SSIM"), p(f'{float(before["estimated_ssim"]):.3f}', "MRICenter"), p(f'{float(after["estimated_ssim"]):.3f}', "MRICenter")],
        [p("Estimated PSNR"), p(f'{before["estimated_psnr_db"]:.2f} dB', "MRICenter"), p(f'{after["estimated_psnr_db"]:.2f} dB', "MRICenter")],
    ], colWidths=[width*0.36, width*0.32, width*0.32])
    overview.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), pale), ("BOX", (0, 0), (-1, -1), 0.7, line),
                                  ("LINEBELOW", (0, 0), (-1, 0), 0.7, line), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                                  ("LEFTPADDING", (0, 0), (-1, -1), 10), ("RIGHTPADDING", (0, 0), (-1, -1), 10),
                                  ("TOPPADDING", (0, 0), (-1, -1), 7), ("BOTTOMPADDING", (0, 0), (-1, -1), 7)]))
    story.append(overview)
    story.append(p("Model performance on held-out test images", "MRIHeading"))
    if model_accuracy is not None:
        accuracy = float(model_accuracy["accuracy_pct"])
        balanced = float(model_accuracy["balanced_accuracy_pct"])
        if not math.isfinite(accuracy) or not 0 <= accuracy <= 100:
            raise ValueError("Report accuracy must be finite and between 0 and 100 percent.")
        if float(model_accuracy["medium"]) != medium or float(model_accuracy["good"]) != good:
            raise ValueError("Model evaluation uses different quality thresholds. Recompute accuracy using this upload's quality bands.")
        model_grade = model_performance_grade(balanced, model_accuracy["n_classes_present"], model_accuracy["n_subjects"])
        model_table = Table([
            [p("Model accuracy", "MRICenter"), p("Balanced accuracy", "MRICenter"), p("Model quality*", "MRICenter")],
            [p(f"{accuracy:.2f}%", "MRIGrade"), p(f"{balanced:.2f}%", "MRIGrade"), p(escape(model_grade), "MRICenter")],
        ], colWidths=[width/3]*3)
        model_table.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), pale), ("BOX", (0, 0), (-1, -1), 0.7, line),
                                         ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                                         ("TOPPADDING", (0, 0), (-1, -1), 6), ("BOTTOMPADDING", (0, 0), (-1, -1), 6)]))
        story.append(model_table)
        story.append(Spacer(1, 5))
        story.append(p(f'{int(model_accuracy["n_candidates"])} input candidates from {int(model_accuracy["n_subjects"])} held-out subjects; '
                       f'{int(model_accuracy["n_classes_present"])}/3 reference-derived quality classes represented. '
                       'This is model-wide test accuracy, not confidence in this uploaded image.', "MRISmall"))
        story.append(p("*Project model-quality bands use balanced accuracy: Good &gt;= 90%, Medium &gt;= 75%, Bad &lt; 75%. "
                       "A grade requires all 3 classes and at least 2 subjects; otherwise: Limited evaluation.", "MRISmall"))
    else:
        story.append(p("Model accuracy: <b>Not available</b> | Model quality: <b>Not evaluated</b>. "
                       + escape(accuracy_note or "Run Step 7's held-out evaluation, then regenerate this report."), "MRISmall"))
    story.append(p("Visual comparison", "MRIHeading"))
    image_side = 175
    comparison = Table([
        [p("Uploaded MRI", "MRICenter"), p("Enhanced MRI", "MRICenter")],
        [Image(_pdf_image_buffer(upload_result["input"]), width=image_side, height=image_side),
         Image(_pdf_image_buffer(upload_result["enhanced"]), width=image_side, height=image_side)],
    ], colWidths=[width/2, width/2])
    comparison.setStyle(TableStyle([("ALIGN", (0, 0), (-1, -1), "CENTER"),
                                    ("VALIGN", (0, 0), (-1, -1), "TOP"), ("BOTTOMPADDING", (0, 0), (-1, 0), 7)]))
    story.append(comparison)
    change = float(after["estimated_ssim"]) - float(before["estimated_ssim"])
    shape = getattr(upload_result["input"], "shape", ())
    resolution = f"{shape[1]} x {shape[0]} pixels" if len(shape) == 2 else "model input resolution"
    story.append(p(f"Pipeline processing time: <b>{milliseconds:.1f} ms</b> | Assessed at {escape(resolution)} | "
                   f"Estimated SSIM change: <b>{change:+.3f}</b>.", "MRISmall"))
    story.append(p("Interpretation", "MRIHeading"))
    story.append(p(f'The uploaded image is rated <b>{before["quality_label"]}</b>; the enhanced image is rated '
                   f'<b>{after["quality_label"]}</b>. These results use the model\'s estimated SSIM, with '
                   f'<b>Bad &lt; {medium:g}</b>, <b>Medium {medium:g} to &lt; {good:g}</b>, '
                   f'and <b>Good &gt;= {good:g}</b>.'))
    reference_available = "before_measured" in report and "after_measured" in report
    if reference_available:
        measured_before, measured_after = report["before_measured"], report["after_measured"]
        story.append(p(f'Reference-based results: before PSNR {float(measured_before["psnr_db"]):.2f} dB, '
                       f'SSIM {float(measured_before["ssim"]):.3f}; after PSNR {float(measured_after["psnr_db"]):.2f} dB, '
                       f'SSIM {float(measured_after["ssim"]):.3f}.', "MRISmall"))
    else:
        story.append(p("No aligned reference scan was supplied. True PSNR/SSIM and model accuracy cannot be measured from this upload alone.", "MRISmall"))
    story.append(p("Scope: brain MRI quality assessment with synthetic-degradation training. Labels are editable project bands, "
                   "not radiologist quality ratings or disease findings. Model-quality bands are project conventions, not clinical validation.", "MRISmall"))

    def footer(canvas, document):
        canvas.saveState()
        canvas.setStrokeColor(line)
        canvas.line(42, 34, A4[0] - 42, 34)
        canvas.setFillColor(slate)
        canvas.setFont("Helvetica", 7)
        canvas.drawString(42, 22, "Group 06 | Research prototype | Estimated image quality")
        canvas.drawRightString(A4[0] - 42, 22, f"Page {document.page}")
        canvas.restoreState()

    document = SimpleDocTemplate(str(output_path), pagesize=A4, rightMargin=42, leftMargin=42,
                                 topMargin=32, bottomMargin=43, title="Uploaded MRI Quality Report", author="Group 06")
    document.build(story, onFirstPage=footer, onLaterPages=footer)
    return output_path


def current_upload_for_report(namespace=None):
    state = globals() if namespace is None else namespace
    for key in ("last_upload_result", "result"):
        candidate = state.get(key)
        if isinstance(candidate, dict) and all(name in candidate for name in ("report", "input", "enhanced", "folder")):
            return candidate
    raise RuntimeError("No analyzed upload is available. Run Step 8, select an MRI and wait for its result, then rerun this report cell.")


# Generate from your current upload without rerunning inference or training.
DOWNLOAD_PDF_REPORT = True
try:
    _report_upload = current_upload_for_report()
except RuntimeError as error:
    print(error)
else:
    if importlib.util.find_spec("reportlab") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "reportlab>=4,<5"])
    _evaluation_folder = globals().get("run_dir")
    if _evaluation_folder is None and Path(_report_upload["folder"]).parent.name == "user_uploads":
        _evaluation_folder = Path(_report_upload["folder"]).parent.parent
    report_model_accuracy, report_accuracy_note = None, None
    if _evaluation_folder is not None:
        _image_bands = _report_upload["report"].get("quality_bands", {})
        try:
            report_model_accuracy = get_report_model_accuracy(_evaluation_folder,
                _image_bands.get("medium_at_or_above", 0.60), _image_bands.get("good_at_or_above", 0.80))
        except (OSError, ValueError) as error:
            report_accuracy_note = str(error)
            print("Accuracy unavailable:", error)
    uploaded_pdf_path = create_uploaded_mri_pdf(_report_upload, model_accuracy=report_model_accuracy,
                                               accuracy_note=report_accuracy_note)
    if report_model_accuracy is not None:
        (Path(_report_upload["folder"]) / "pdf_model_evaluation.json").write_text(json.dumps(report_model_accuracy, indent=2), encoding="utf-8")
        print(f"Model accuracy: {report_model_accuracy['accuracy_pct']:.2f}%")
        print("Model quality:", report_model_accuracy["model_quality"])
    print("PDF report saved:", uploaded_pdf_path)
    print("Report generated for:", _report_upload["report"].get("uploaded_filename", "Uploaded MRI"))
    if DOWNLOAD_PDF_REPORT:
        try:
            from google.colab import files
        except ImportError:
            print("Outside Colab: open the saved PDF path above.")
        else:
            files.download(str(uploaded_pdf_path))
'''
FINAL_STEP_SOURCES['uploaded_mri_report.py'] = _stage_source
exec(compile(_stage_source, 'uploaded_mri_report.py', "exec"), globals())

## 10. Download the project results

Export the selected model, source, settings, manifest, metrics, figures and uploaded
image reports/PDFs. Original NIfTI volumes are excluded. Download this notebook with
**File → Download → Download .ipynb** to retain new outputs. For another MRI later,
rerun only Steps 8 and 9.

In [ ]:
#@title 10. Package and download results
import shutil
import zipfile
from pathlib import Path

if not globals().get('QA_TEST_METRICS') or Path(run_dir) != Path(QA_UPGRADE_DIR):
    raise RuntimeError('Complete Step 7 before exporting the selected model.')
source_dir = Path(run_dir) / 'source'
source_dir.mkdir(parents=True, exist_ok=True)
(source_dir / 'mri_quality_final.py').write_text(FINAL_MODEL_SOURCE, encoding='utf-8')
for filename, source in FINAL_STEP_SOURCES.items():
    (source_dir / filename).write_text(source, encoding='utf-8')

baseline_dir = Path(run_dir) / 'baseline_training'
baseline_dir.mkdir(parents=True, exist_ok=True)
for filename in ('enhancer_best.pt', 'enhancer_last.pt', 'quality_best.pt',
                 'quality_last.pt', 'enhancer_history.csv', 'quality_history.csv',
                 'config.json', 'subject_manifest.json'):
    source = Path(QA_SOURCE_RUN_DIR) / filename
    if source.is_file():
        shutil.copy2(source, baseline_dir / filename)

(Path(run_dir) / 'LOAD_MODEL.txt').write_text(
    'Selected inference weights are in this folder.\n'
    'Add source/ to sys.path, then:\n'
    'from mri_quality_final import MRIPipeline, load_demo_image\n'
    'pipeline = MRIPipeline.from_folder("PATH_TO_THIS_FOLDER")\n'
    'image = load_demo_image("my_mri.png", pipeline.image_size)\n'
    'enhanced, report = pipeline.process(image)\n'
    'print(report)\n\n'
    'The final notebook contains the complete training workflow.\n'
    'baseline_training/ preserves warm-start checkpoints.\n'
    'Top-level quality_best.pt is the selected model.\n', encoding='utf-8')

archive_dir = Path(cfg.root) / 'deliverables'
archive_dir.mkdir(parents=True, exist_ok=True)
archive_path = archive_dir / f'MRI_Group06_Final_{Path(run_dir).name}.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(Path(run_dir).rglob('*')):
        if path.is_file() and not path.name.lower().endswith(('.nii', '.nii.gz', '.tmp')):
            archive.write(path, arcname='MRI_Group06_Final/' + path.relative_to(run_dir).as_posix())
print('Saved:', archive_path)
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(archive_path)))
else:
    files.download(str(archive_path))

### References and interpretation

- [OpenNeuro ds000221](https://openneuro.org/datasets/ds000221) and
  [dataset citations](https://github.com/OpenNeuroDatasets/ds000221).
- [U-Net paper](https://arxiv.org/abs/1505.04597).
- [PSNR/SSIM documentation](https://scikit-image.org/docs/stable/api/skimage.metrics.html).

Acquired T1 MRI is the research reference. Synthetic noise/downsampling approximates
selected low-quality conditions, not complete scanner physics. SSIM-derived quality
bands are project conventions, not radiologist ratings or disease findings. Slices
and simulated variants from one subject are correlated. Previous test subjects that
guided tuning require additional unseen subjects for a final generalization claim.
Performance is measured when you run the notebook; no accuracy is guaranteed.

In [ ]:
%%writefile index.html
<!DOCTYPE html>
<html lang="en" class="dark">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Group 06 | Cranial MRI IQA & U-Net Enhancement Workstation</title>
  <script src="https://cdn.tailwindcss.com"></script>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500;600;700&display=swap" rel="stylesheet">
  <style>
    body { font-family: 'Inter', sans-serif; background-color: #060911; color: #F8FAFC; }
    .mono { font-family: 'JetBrains Mono', monospace; }
    .glass { background: rgba(15, 23, 42, 0.78); backdrop-filter: blur(16px); border: 1px solid rgba(148, 163, 184, 0.12); }
    .comparison-slider { position: relative; overflow: hidden; user-select: none; }
    .comparison-overlay { position: absolute; top: 0; left: 0; height: 100%; width: 50%; overflow: hidden; }
    input[type=range]::-webkit-slider-thumb { cursor: pointer; }
  </style>
</head>
<body class="h-screen flex flex-col overflow-hidden select-none">

  <!-- Top PACS Header -->
  <header class="h-16 glass px-6 flex items-center justify-between border-b border-slate-800/80 z-20">
    <div class="flex items-center gap-3.5">
      <div class="w-10 h-10 rounded-xl bg-gradient-to-br from-cyan-500/20 to-blue-600/20 border border-cyan-400/30 flex items-center justify-center text-cyan-400 font-bold mono shadow-inner">
        G06
      </div>
      <div>
        <div class="flex items-center gap-2.5">
          <h1 class="text-sm font-bold tracking-wide text-white">CRANIAL MRI QUALITY & ENHANCEMENT</h1>
          <span class="text-[10px] mono px-2 py-0.5 rounded-md bg-cyan-500/15 text-cyan-300 border border-cyan-500/30">RESIDUAL U-NET + QUALITYNET</span>
        </div>
        <p class="text-[11px] text-slate-400">Automated No-Reference PSNR/SSIM Regression & Artifact Restoration</p>
      </div>
    </div>

    <!-- Held-Out Model Benchmark Pill -->
    <div class="hidden xl:flex items-center gap-6 px-4 py-1.5 rounded-xl bg-slate-900/90 border border-slate-800 text-xs">
      <div>
        <span class="text-slate-500">Held-Out Test Acc:</span>
        <span id="hdr-acc" class="mono font-semibold text-white ml-1">—</span>
      </div>
      <div class="h-3 w-[1px] bg-slate-800"></div>
      <div>
        <span class="text-slate-500">Balanced Acc:</span>
        <span id="hdr-bal" class="mono font-semibold text-cyan-400 ml-1">—</span>
      </div>
      <div class="h-3 w-[1px] bg-slate-800"></div>
      <div>
        <span class="text-slate-500">Model Grade:</span>
        <span id="hdr-grade" class="mono font-semibold text-emerald-400 ml-1">—</span>
      </div>
    </div>

    <div class="flex items-center gap-3">
      <span id="status-pill" class="text-[11px] mono px-3 py-1.5 rounded-full bg-slate-900 text-slate-400 border border-slate-800">
        ● PIPELINE READY
      </span>
      <label class="cursor-pointer bg-gradient-to-r from-cyan-500 to-blue-500 hover:from-cyan-400 hover:to-blue-400 text-slate-950 font-semibold text-xs px-4 py-2.5 rounded-xl transition shadow-lg shadow-cyan-500/20 flex items-center gap-2">
        <span>+ Upload MRI (.PNG / .NII.GZ)</span>
        <input type="file" id="file-input" accept=".png,.jpg,.jpeg,.bmp,.tif,.tiff,.nii,.nii.gz" class="hidden" onchange="uploadScan(event)">
      </label>
    </div>
  </header>

  <!-- Main Radiology Grid -->
  <main class="flex-1 grid grid-cols-12 gap-4 p-4 overflow-hidden">

    <!-- Left Panel: Controls & EDA Slice Telemetry -->
    <aside class="col-span-3 glass rounded-2xl p-5 flex flex-col justify-between overflow-y-auto">
      <div class="space-y-6">
        <!-- Viewport Mode Switcher -->
        <div>
          <h2 class="text-[11px] font-semibold uppercase tracking-wider text-slate-400 mb-2.5">Diagnostic Viewport Mode</h2>
          <div class="grid grid-cols-3 gap-1.5 p-1 rounded-xl bg-slate-950/80 border border-slate-800">
            <button onclick="setViewMode('split')" id="mode-split" class="text-xs py-2 rounded-lg bg-cyan-500/20 text-cyan-300 border border-cyan-500/30 font-medium transition">
              Split Curtain
            </button>
            <button onclick="setViewMode('side')" id="mode-side" class="text-xs py-2 rounded-lg text-slate-400 hover:text-white transition">
              Side-by-Side
            </button>
            <button onclick="setViewMode('error')" id="mode-error" class="text-xs py-2 rounded-lg text-slate-400 hover:text-white transition">
              Magma Diff
            </button>
          </div>
        </div>

        <!-- PACS Windowing Sliders -->
        <div class="border-t border-slate-800/80 pt-5">
          <div class="flex justify-between items-center mb-3">
            <h2 class="text-[11px] font-semibold uppercase tracking-wider text-slate-400">Radiological Windowing</h2>
            <button onclick="resetWindow()" class="text-[10px] text-cyan-400 hover:underline">Reset</button>
          </div>
          <div class="space-y-3.5">
            <div>
              <div class="flex justify-between text-xs mb-1">
                <span class="text-slate-400">Brightness (Level)</span>
                <span id="lbl-bright" class="mono text-slate-300">100%</span>
              </div>
              <input type="range" id="rng-bright" min="50" max="160" value="100" oninput="applyWindow()" class="w-full accent-cyan-400 h-1.5 bg-slate-800 rounded-lg">
            </div>
            <div>
              <div class="flex justify-between text-xs mb-1">
                <span class="text-slate-400">Contrast (Width)</span>
                <span id="lbl-contrast" class="mono text-slate-300">100%</span>
              </div>
              <input type="range" id="rng-contrast" min="50" max="180" value="100" oninput="applyWindow()" class="w-full accent-cyan-400 h-1.5 bg-slate-800 rounded-lg">
            </div>
          </div>
        </div>

        <!-- Step 3a Engineered Slice Statistics -->
        <div class="border-t border-slate-800/80 pt-5">
          <h2 class="text-[11px] font-semibold uppercase tracking-wider text-slate-400 mb-3">Slice Telemetry (EDA Features)</h2>
          <div class="space-y-2 text-xs bg-slate-950/60 p-3.5 rounded-xl border border-slate-800/80">
            <div class="flex justify-between"><span class="text-slate-500">File Name:</span> <span id="meta-file" class="mono text-slate-200 truncate max-w-[140px]">—</span></div>
            <div class="flex justify-between"><span class="text-slate-500">Assessed Grid:</span> <span id="meta-res" class="mono text-slate-200">—</span></div>
            <div class="flex justify-between"><span class="text-slate-500">Inference Time:</span> <span id="meta-ms" class="mono text-emerald-400">—</span></div>
            <div class="border-t border-slate-800/80 my-1.5"></div>
            <div class="flex justify-between"><span class="text-slate-500">Foreground Tissue:</span> <span id="eda-fg" class="mono text-slate-300">—</span></div>
            <div class="flex justify-between"><span class="text-slate-500">Intensity Entropy:</span> <span id="eda-ent" class="mono text-slate-300">—</span></div>
            <div class="flex justify-between"><span class="text-slate-500">Laplacian Var (Sharpness):</span> <span id="eda-lap" class="mono text-cyan-300">—</span></div>
            <div class="flex justify-between"><span class="text-slate-500">HF Residual Median:</span> <span id="eda-hf" class="mono text-slate-300">—</span></div>
          </div>
        </div>
      </div>

      <!-- Action Buttons (PNG + Step 9 Official PDF Report) -->
      <div class="space-y-2.5 pt-4 border-t border-slate-800/80">
        <button id="btn-pdf" disabled onclick="downloadPDF()" class="w-full py-2.5 px-4 rounded-xl bg-slate-800 text-slate-500 text-xs font-semibold border border-slate-700 transition flex items-center justify-center gap-2">
          <span>📄 Download Official PDF Report</span>
        </button>
        <button id="btn-png" disabled onclick="downloadPNG()" class="w-full py-2 px-4 rounded-xl bg-slate-900 text-slate-500 text-xs font-medium border border-slate-800 transition">
          ⬇ Export Enhanced Slice (.PNG)
        </button>
      </div>
    </aside>

    <!-- Center Panel: Interactive MRI Canvas -->
    <section class="col-span-6 glass rounded-2xl relative flex flex-col items-center justify-center overflow-hidden bg-[#030509]">
      <!-- Empty State -->
      <div id="empty-state" class="text-center p-8 max-w-sm">
        <div class="w-16 h-16 rounded-2xl bg-cyan-500/10 border border-cyan-500/20 mx-auto flex items-center justify-center mb-4 text-2xl shadow-inner">
          🧠
        </div>
        <h3 class="text-base font-semibold text-slate-200">No Cranial MRI Loaded</h3>
        <p class="text-xs text-slate-400 mt-1.5 leading-relaxed">
          Upload a 2D axial slice (<span class="mono text-slate-300">.png, .jpg, .tif</span>) or a 3D T1 NIfTI volume (<span class="mono text-slate-300">.nii, .nii.gz</span>). NIfTI uploads automatically extract and pad the canonical mid-axial slice.
        </p>
      </div>

      <!-- Mode 1: Split Curtain Comparison -->
      <div id="view-split" class="hidden relative w-full h-full flex items-center justify-center p-6">
        <div class="comparison-slider relative rounded-xl overflow-hidden border border-slate-800 shadow-2xl">
          <img id="split-after" class="block h-[68vh] w-[68vh] object-contain bg-black" src="" alt="Enhanced">
          <div id="split-clip" class="comparison-overlay border-r-2 border-cyan-400">
            <img id="split-before" class="block h-[68vh] w-[68vh] max-w-none object-contain bg-black" src="" alt="Uploaded">
          </div>
          <span class="absolute top-3 left-3 bg-slate-950/85 border border-slate-700 text-[10px] mono px-2.5 py-1 rounded-md text-slate-300">UPLOADED MRI</span>
          <span class="absolute top-3 right-3 bg-cyan-950/90 border border-cyan-500/40 text-[10px] mono px-2.5 py-1 rounded-md text-cyan-300">AFTER U-NET</span>
        </div>
        <input type="range" id="curtain-slider" min="0" max="100" value="50" oninput="moveCurtain(this.value)" class="absolute inset-0 w-full h-full opacity-0 cursor-ew-resize z-20">
      </div>

      <!-- Mode 2: Side-by-Side -->
      <div id="view-side" class="hidden w-full h-full grid grid-cols-2 gap-4 p-6 items-center">
        <div class="relative rounded-xl overflow-hidden border border-slate-800 bg-black flex flex-col items-center">
          <img id="side-before" class="w-full h-[58vh] object-contain" src="">
          <span class="absolute top-3 left-3 bg-slate-950/85 border border-slate-700 text-[10px] mono px-2.5 py-1 rounded text-slate-300">UPLOADED INPUT</span>
        </div>
        <div class="relative rounded-xl overflow-hidden border border-cyan-500/30 bg-black flex flex-col items-center">
          <img id="side-after" class="w-full h-[58vh] object-contain" src="">
          <span class="absolute top-3 right-3 bg-cyan-950/90 border border-cyan-500/40 text-[10px] mono px-2.5 py-1 rounded text-cyan-300">ENHANCED (U-NET)</span>
        </div>
      </div>

      <!-- Mode 3: Magma Residual Correction Map -->
      <div id="view-error" class="hidden w-full h-full flex flex-col items-center justify-center p-6">
        <div class="relative rounded-xl overflow-hidden border border-amber-500/30 bg-black">
          <img id="img-magma" class="h-[68vh] w-[68vh] object-contain" src="">
          <span class="absolute top-3 left-3 bg-slate-950/90 border border-amber-500/40 text-[10px] mono px-2.5 py-1 rounded text-amber-300">ABSOLUTE U-NET CORRECTION (|AFTER − BEFORE|)</span>
        </div>
      </div>
    </section>

    <!-- Right Panel: QualityNet Predictions & Quality Bands -->
    <aside class="col-span-3 glass rounded-2xl p-5 flex flex-col justify-between overflow-y-auto">
      <div class="space-y-4">
        <h2 class="text-[11px] font-semibold uppercase tracking-wider text-slate-400">QualityNet Classification</h2>

        <!-- Before vs After Quality Band Badges -->
        <div class="grid grid-cols-2 gap-2.5">
          <div class="p-3.5 rounded-xl bg-slate-900/90 border border-slate-800">
            <span class="text-[10px] uppercase tracking-wider text-slate-400 block">Uploaded Band</span>
            <span id="badge-before" class="text-xl font-bold mono mt-1 block text-slate-500">—</span>
          </div>
          <div class="p-3.5 rounded-xl bg-slate-900/90 border border-slate-800">
            <span class="text-[10px] uppercase tracking-wider text-slate-400 block">Enhanced Band</span>
            <span id="badge-after" class="text-xl font-bold mono mt-1 block text-slate-500">—</span>
          </div>
        </div>

        <!-- Estimated SSIM Card -->
        <div class="p-4 rounded-xl bg-slate-900/90 border border-slate-800 space-y-2.5">
          <div class="flex justify-between items-center">
            <span class="text-xs text-slate-400 font-medium">Estimated SSIM</span>
            <span id="ssim-delta" class="text-[11px] mono px-2 py-0.5 rounded bg-emerald-500/10 text-emerald-400 border border-emerald-500/20">—</span>
          </div>
          <div class="flex items-baseline justify-between">
            <div>
              <span class="text-[10px] text-slate-500 block">AFTER U-NET</span>
              <span id="ssim-after" class="text-2xl font-bold mono text-white">—</span>
            </div>
            <div class="text-right">
              <span class="text-[10px] text-slate-500 block">UPLOADED</span>
              <span id="ssim-before" class="text-sm mono text-slate-400">—</span>
            </div>
          </div>
          <!-- Threshold Progress Bar (0.60 Medium, 0.80 Good) -->
          <div class="relative w-full bg-slate-800 h-2 rounded-full overflow-hidden">
            <div id="ssim-bar" class="bg-gradient-to-r from-cyan-500 to-emerald-400 h-full w-0 transition-all duration-500"></div>
          </div>
          <div class="flex justify-between text-[10px] mono text-slate-500">
            <span>Bad (&lt;0.60)</span>
            <span>Med (≥0.60)</span>
            <span>Good (≥0.80)</span>
          </div>
        </div>

        <!-- Estimated PSNR Card -->
        <div class="p-4 rounded-xl bg-slate-900/90 border border-slate-800 space-y-2.5">
          <div class="flex justify-between items-center">
            <span class="text-xs text-slate-400 font-medium">Estimated PSNR (dB)</span>
            <span id="psnr-delta" class="text-[11px] mono px-2 py-0.5 rounded bg-cyan-500/10 text-cyan-300 border border-cyan-500/20">—</span>
          </div>
          <div class="flex items-baseline justify-between">
            <div>
              <span class="text-[10px] text-slate-500 block">AFTER U-NET</span>
              <span id="psnr-after" class="text-2xl font-bold mono text-white">—</span>
            </div>
            <div class="text-right">
              <span class="text-[10px] text-slate-500 block">UPLOADED</span>
              <span id="psnr-before" class="text-sm mono text-slate-400">—</span>
            </div>
          </div>
          <div class="w-full bg-slate-800 h-2 rounded-full overflow-hidden">
            <div id="psnr-bar" class="bg-cyan-400 h-full w-0 transition-all duration-500"></div>
          </div>
        </div>
      </div>

      <!-- Research Scope Notice -->
      <div class="p-3.5 rounded-xl bg-slate-900/70 border border-slate-800/90 text-[11px] text-slate-400 leading-relaxed space-y-1">
        <div class="text-slate-300 font-semibold">Scope & Thresholds</div>
        <p>Bands derive from predicted SSIM (<span class="mono text-emerald-400">Good ≥ 0.80</span>, <span class="mono text-amber-400">Medium ≥ 0.60</span>). Trained on OpenNeuro ds000221 T1 cranial MRI with synthetic Gaussian/Rician degradation.</p>
      </div>
    </aside>
  </main>

  <script>
    let currentResult = null;
    const bandColors = {
      "Good": "text-emerald-400",
      "Medium": "text-amber-400",
      "Bad": "text-rose-400"
    };

    // Fetch held-out test metrics on startup
    window.addEventListener('DOMContentLoaded', async () => {
      try {
        const res = await fetch('/api/model-info');
        const info = await res.json();
        if (info.accuracy_pct) {
          document.getElementById('hdr-acc').innerText = info.accuracy_pct.toFixed(2) + '%';
          document.getElementById('hdr-bal').innerText = info.balanced_accuracy_pct.toFixed(2) + '%';
          document.getElementById('hdr-grade').innerText = info.model_quality;
        }
      } catch (e) {}
    });

    async function uploadScan(event) {
      const file = event.target.files[0];
      if (!file) return;

      const pill = document.getElementById('status-pill');
      pill.innerText = "● RUNNING U-NET & QUALITYNET...";
      pill.className = "text-[11px] mono px-3 py-1.5 rounded-full bg-amber-500/20 text-amber-300 border border-amber-500/40";

      const formData = new FormData();
      formData.append('file', file);

      try {
        const response = await fetch('/api/analyze', { method: 'POST', body: formData });
        const data = await response.json();
        if (!response.ok) throw new Error(data.detail || "Analysis failed");
        currentResult = data;
        renderResult(data);

        pill.innerText = "● ANALYSIS COMPLETE";
        pill.className = "text-[11px] mono px-3 py-1.5 rounded-full bg-emerald-500/20 text-emerald-300 border border-emerald-500/40";
      } catch (err) {
        alert("Upload Error: " + err.message);
        pill.innerText = "● ERROR";
        pill.className = "text-[11px] mono px-3 py-1.5 rounded-full bg-rose-500/20 text-rose-300 border border-rose-500/40";
      }
    }

    function renderResult(data) {
      document.getElementById('empty-state').classList.add('hidden');
      setViewMode('split');

      // Populate Images
      ['split-before', 'side-before'].forEach(id => document.getElementById(id).src = data.input_b64);
      ['split-after', 'side-after'].forEach(id => document.getElementById(id).src = data.enhanced_b64);
      document.getElementById('img-magma').src = data.magma_b64;

      const r = data.report;
      const bef = r.before_estimated;
      const aft = r.after_estimated;

      // Badges
      const bEl = document.getElementById('badge-before');
      const aEl = document.getElementById('badge-after');
      bEl.innerText = bef.quality_label;
      bEl.className = `text-xl font-bold mono mt-1 block ${bandColors[bef.quality_label]}`;
      aEl.innerText = aft.quality_label;
      aEl.className = `text-xl font-bold mono mt-1 block ${bandColors[aft.quality_label]}`;

      // SSIM
      const ssimGain = aft.estimated_ssim - bef.estimated_ssim;
      document.getElementById('ssim-before').innerText = bef.estimated_ssim.toFixed(3);
      document.getElementById('ssim-after').innerText = aft.estimated_ssim.toFixed(3);
      document.getElementById('ssim-delta').innerText = (ssimGain >= 0 ? '+' : '') + ssimGain.toFixed(3);
      document.getElementById('ssim-bar').style.width = `${Math.max(0, Math.min(100, aft.estimated_ssim * 100))}%`;

      // PSNR
      const psnrGain = aft.estimated_psnr_db - bef.estimated_psnr_db;
      document.getElementById('psnr-before').innerText = bef.estimated_psnr_db.toFixed(2) + ' dB';
      document.getElementById('psnr-after').innerText = aft.estimated_psnr_db.toFixed(2) + ' dB';
      document.getElementById('psnr-delta').innerText = (psnrGain >= 0 ? '+' : '') + psnrGain.toFixed(2) + ' dB';
      document.getElementById('psnr-bar').style.width = `${Math.max(0, Math.min(100, (aft.estimated_psnr_db / 45) * 100))}%`;

      // Telemetry & Step 3a EDA stats
      document.getElementById('meta-file').innerText = r.uploaded_filename;
      document.getElementById('meta-res').innerText = `${r.assessed_resolution[0]} × ${r.assessed_resolution[1]} px`;
      document.getElementById('meta-ms').innerText = `${r.pipeline_ms.toFixed(1)} ms`;
      document.getElementById('eda-fg').innerText = (data.eda.foreground_fraction * 100).toFixed(1) + '%';
      document.getElementById('eda-ent').innerText = data.eda.intensity_entropy.toFixed(3);
      document.getElementById('eda-lap').innerText = `${data.eda.lap_before.toFixed(4)} → ${data.eda.lap_after.toFixed(4)}`;
      document.getElementById('eda-hf').innerText = data.eda.high_frequency_residual_median.toFixed(4);

      // Enable Downloads
      const btnPdf = document.getElementById('btn-pdf');
      btnPdf.disabled = false;
      btnPdf.className = "w-full py-2.5 px-4 rounded-xl bg-gradient-to-r from-cyan-500 to-blue-500 hover:from-cyan-400 hover:to-blue-400 text-slate-950 text-xs font-bold transition flex items-center justify-center gap-2 shadow-lg shadow-cyan-500/20 cursor-pointer";

      const btnPng = document.getElementById('btn-png');
      btnPng.disabled = false;
      btnPng.className = "w-full py-2 px-4 rounded-xl bg-slate-800 hover:bg-slate-700 text-slate-200 text-xs font-medium border border-slate-700 transition cursor-pointer";
    }

    function moveCurtain(val) {
      document.getElementById('split-clip').style.width = val + '%';
    }

    function setViewMode(mode) {
      if (!currentResult) return;
      ['split', 'side', 'error'].forEach(m => {
        document.getElementById('view-' + m).classList.add('hidden');
        document.getElementById('mode-' + m).className = "text-xs py-2 rounded-lg text-slate-400 hover:text-white transition";
      });
      document.getElementById('view-' + mode).classList.remove('hidden');
      document.getElementById('mode-' + mode).className = "text-xs py-2 rounded-lg bg-cyan-500/20 text-cyan-300 border border-cyan-500/30 font-medium transition";
    }

    function applyWindow() {
      const b = document.getElementById('rng-bright').value;
      const c = document.getElementById('rng-contrast').value;
      document.getElementById('lbl-bright').innerText = b + '%';
      document.getElementById('lbl-contrast').innerText = c + '%';
      const f = `brightness(${b}%) contrast(${c}%)`;
      ['split-before', 'split-after', 'side-before', 'side-after'].forEach(id => {
        document.getElementById(id).style.filter = f;
      });
    }

    function resetWindow() {
      document.getElementById('rng-bright').value = 100;
      document.getElementById('rng-contrast').value = 100;
      applyWindow();
    }

    function downloadPNG() {
      if (!currentResult) return;
      const a = document.createElement('a');
      a.href = currentResult.enhanced_b64;
      a.download = 'Group06_Enhanced_' + currentResult.report.uploaded_filename + '.png';
      a.click();
    }

    function downloadPDF() {
      window.open('/api/download-pdf', '_blank');
    }
  </script>
</body>
</html>

In [ ]:
!pip install -q fastapi uvicorn python-multipart nest-asyncio pycloudflared reportlab

import base64
import threading
import cv2
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
import matplotlib.cm as cm
import nest_asyncio
import numpy as np
from pycloudflared import try_cloudflare
import uvicorn

nest_asyncio.apply()
app = FastAPI(title="Group 06 MRI Workstation")


def array_to_b64(arr_01: np.ndarray) -> str:
    """Converts a [0,1] float32 slice into a Base64 PNG."""
    uint8_img = (np.clip(arr_01, 0, 1) * 255).round().astype(np.uint8)
    _, buf = cv2.imencode(".png", uint8_img)
    return f"data:image/png;base64,{base64.b64encode(buf).decode('utf-8')}"


def magma_diff_b64(before_01: np.ndarray, after_01: np.ndarray) -> str:
    """Generates the Magma residual error map matching Step 7 plot_comparisons."""
    diff = np.clip(np.abs(after_01 - before_01) / 0.25, 0, 1)
    rgba = (cm.get_cmap("magma")(diff)[..., :3] * 255).round().astype(np.uint8)
    bgr = cv2.cvtColor(rgba, cv2.COLOR_RGB2BGR)
    _, buf = cv2.imencode(".png", bgr)
    return f"data:image/png;base64,{base64.b64encode(buf).decode('utf-8')}"


@app.get("/", response_class=HTMLResponse)
async def index():
    return Path("index.html").read_text(encoding="utf-8")


@app.get("/api/model-info")
async def model_info():
    # Uses your Step 7 QA_TEST_METRICS if available
    metrics = globals().get("QA_TEST_METRICS", {}).get("Before (degraded)", {})
    return JSONResponse(metrics)


@app.post("/api/analyze")
async def analyze_mri(file: UploadFile = File(...)):
    global last_upload_result
    content = await file.read()
    try:
        # Calls your exact Step 8 notebook function!
        result = analyze_uploaded_file({file.filename: content})
        last_upload_result = result
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

    # Compute Step 3a EDA statistics on the uploaded and enhanced slice
    eda_in = (
        _eda_feature_row(result["input"])
        if "_eda_feature_row" in globals()
        else {}
    )
    eda_out = (
        _eda_feature_row(result["enhanced"])
        if "_eda_feature_row" in globals()
        else {}
    )

    return JSONResponse({
        "input_b64": array_to_b64(result["input"]),
        "enhanced_b64": array_to_b64(result["enhanced"]),
        "magma_b64": magma_diff_b64(result["input"], result["enhanced"]),
        "report": result["report"],
        "eda": {
            "foreground_fraction": eda_in.get("foreground_fraction", 0),
            "intensity_entropy": eda_in.get("intensity_entropy", 0),
            "high_frequency_residual_median": eda_in.get(
                "high_frequency_residual_median", 0
            ),
            "lap_before": eda_in.get("laplacian_variance", 0),
            "lap_after": eda_out.get("laplacian_variance", 0),
        },
    })


@app.get("/api/download-pdf")
async def download_pdf():
    if not globals().get("last_upload_result"):
        raise HTTPException(status_code=400, detail="Analyze an MRI first.")
    acc = None
    if "get_report_model_accuracy" in globals() and "run_dir" in globals():
        try:
            acc = get_report_model_accuracy(run_dir)
        except Exception:
            pass
    pdf_path = create_uploaded_mri_pdf(last_upload_result, model_accuracy=acc)
    return FileResponse(
        str(pdf_path),
        media_type="application/pdf",
        filename=f"Group06_Report_{last_upload_result['report']['uploaded_filename']}.pdf",
    )


# Launch server in background thread and print public URL
threading.Thread(
    target=lambda: uvicorn.run(
        app, host="127.0.0.1", port=8000, log_level="error"
    ),
    daemon=True,
).start()
public_url = try_cloudflare(port=8000)
print(
    f"\n🚀 OPEN YOUR GROUP 06 PREMIUM WORKSTATION UI HERE: {public_url.tunnel}"
)